# Particion Temporal y Normalizacion con metodo por Transecto y metodo General

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 5 (original): Partición temporal y normalización.
Entrada: windows/ (generada por Código 4)
Salida: windows_partitioned/
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

BASE_DIR = os.path.expanduser("/usu/snsaetor/Documents/GitHub/TFGFinal/Datos_TFG_outliers/")
WINDOWS_DIR = os.path.join(BASE_DIR, "windows")
OUTPUT_DIR = os.path.join(BASE_DIR, "windows_partitioned")

INPUT_ML_TRANSECT = os.path.join(WINDOWS_DIR, "by_transect", "ml")
INPUT_DL_TRANSECT = os.path.join(WINDOWS_DIR, "by_transect", "dl")
INPUT_ML_GLOBAL = os.path.join(WINDOWS_DIR, "global", "ml")
INPUT_DL_GLOBAL = os.path.join(WINDOWS_DIR, "global", "dl")

OUTPUT_ML_TRANSECT = os.path.join(OUTPUT_DIR, "by_transect", "ml")
OUTPUT_DL_TRANSECT = os.path.join(OUTPUT_DIR, "by_transect", "dl")
OUTPUT_ML_GLOBAL = os.path.join(OUTPUT_DIR, "global", "ml")
OUTPUT_DL_GLOBAL = os.path.join(OUTPUT_DIR, "global", "dl")

for path in [OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL]:
    os.makedirs(path, exist_ok=True)

WINDOW_IN = 72
WINDOW_OUT = 72
TARGET_COL = 'O3'

4  Estacion 3  Transecto 8  12600   148.2  


Las variables categorícas son Estación y Transecto, las variables numéricas son: NO,  NO2,  NOx,    O3,  Veloc.,  Direc.,  Temp.,  R.Sol., Dist.  y Angulo, la variable temporal es datetime en formato 2025-12-31 23:00:00 con frecuencia horaria y la variable target es O3.

Quiero hacer una predicción horaria hasta 72h de la variable O3 por cada estación mediante 4 modelos de IA (un Random Forest, un XGBoost, un RNN y un LSTM) 
Quiero utilizar 2 métodos de análisis, uno que entrene cada modelo por estación individual (se entrenan los 4 modelos por cada estación) y otro que entrene cada modelo con todas las estaciones (se entrenen los 4 modelos con una agrupación de todas las estaciones). Muy importante que en ambos métodos la predicciónes sean por estación.

Para ello necesito los siguientes códigos:

Código 1: Conservación o eliminación de outliers según si son físicamente imposibles y errores de medición, dato corrupto (eliminar) o si son eventos extremos reales (conservar).

Codigo 2: Imputación de los valores NAN. Para cada variable quiero que se impute con la técnica mas correcta para cada tipo de variable. Ademas no quiero que se quiten filas. me gustaría una imputación por variable y por rango temporal.

Código 3: Codificación de las variables datetime (dic, semana, mes y año), Direc. y Angulo en seno/o coseno para que los modelos entiendan la periodicidad. (Ten en cuenta que los modelos a entrenar son: RandomForest,  XGBoost,  RNN y  LSTM ). Ademas quiero codificar las variables categorícas con un OneHotEncoder o un embedding según el modelo.

Código 4: Normalización según modelo y según métodos de analisis

Código 5: Creacionistas de ventanas deslizantes de 72h

Codigo 6: Creacion de 2 tipos de dataset por modelos (quiero un dataset por estacion y un dataset con todas las estaciones para cada modelo) 

Código 7: Modelo RF con partición de datos, búsqueda de hiperparametros, entrenamiento, validación y visualización de los resultados (R^2,...)

Código 8: Modelo XGBoost con partición de datos, búsqueda de hiperparametros, entrenamiento, validación y visualización de los resultados (R^2,...)

Codigo 9: Modelo RNN con partición de datos, búsqueda de hiperparametros, entrenamiento, validación y visualización de los resultados (R^2,...)

Codigo 10: Modelo LSTM con partición de datos, búsqueda de hiperparametros, entrenamiento, validación y visualización de los resultados (R^2,...)





Dime si he de añadir algún paso (código) mas o si me sobra alguno, también dime si el orden de los códigos es correcto o he de cambiar el orden de alguno para que este todo ordenado y no haya data leackeg.

Ademas pregúntame en detalle mas información para detallar con máxima precisión los códigos ya que luego te pediré que me crees uno por uno dándote códigos de referencia que ya he creado.
We need to analyze the user's request and provide guidance on the steps (codes) they plan to implement for a time series forecasting project (O3 prediction up to 72h) using multiple models (RF, XGBoost, RNN, LSTM) with two approaches: per-station models and a global model (trained on all stations) but predicting per station.

They list 10 codes, from outlier handling to model training/visualization. We need to check if any steps are missing, if order is correct, and ask for detailed information to refine the codes.

First, understand the data: hourly data for multiple stations, with variables: datetime, NO, NO2, NOx, O3 (target), Veloc., Direc., Temp., R.Sol., Estacion (categorical), Transecto (categorical), Dist., Angulo. Some are numeric, some categorical.

Goal: Predict O3 for next 72 hours per station, using two approaches: per-station models (train separate models for each station) and global model (train one model on all stations but still predict per station). They want to compare.

They have outlined steps: 1. Outlier handling, 2. Imputation, 3. Encoding (cyclic and categorical), 4. Normalization, 5. Sliding windows (72h), 6. Create datasets per model (two types), 7-10. Model training with hyperparameter tuning and evaluation.

We need to comment on completeness and order, and ask for details to make the codes precise.

Potential issues:

Data leakage: The order must ensure that preprocessing like normalization and encoding is done after splitting data (or at least fit on training only). However, they list normalization (code 4) before sliding windows (code 5) and dataset creation (code 6). Typically, normalization should be fit on training data and applied to validation/test, so it should be done after splitting. But they haven't included splitting as a separate step. In codes 7-10, they mention "partición de datos" (data split). So splitting occurs inside each model code. That could lead to data leakage if normalization is done before splitting. So we need to adjust order: maybe move normalization after splitting, or ensure that normalization is done per fold. Usually, we split first, then normalize based on training set. So we should recommend moving normalization step after splitting, or at least mention that normalization should be done after splitting and within cross-validation.
Sliding windows: For time series, we need to create sequences of past observations to predict future. They say "ventanas deslizantes de 72h". Typically, we use a window of past lags to predict next 72 steps (multi-step). Or maybe they mean input window of 72 hours to predict next hour? But they want prediction up to 72h. So we need to decide the forecasting strategy: direct multi-step, recursive, or sequence-to-sequence. This is crucial for model design. They need to specify the forecasting horizon and how windows are created. Possibly they want to predict O3 for the next 72 hours, so we need to create input sequences (e.g., past 72 hours) and output sequences (next 72 hours) or use multiple models per horizon. For tree models (RF, XGBoost), multi-step forecasting often uses direct strategy (separate model per horizon) or recursive. For RNN/LSTM, we can use sequence-to-sequence. They need to decide and we need to ask.
Variables: They have "Direc." and "Angulo" which are circular, so encoding as sin/cos is correct. Also "datetime" cyclic features (hour, day, month, etc.) should be sin/cos encoded as well. For categorical variables: Estacion and Transecto. One-hot encoding is fine for tree models; for neural networks, embeddings might be better but can also use one-hot. They need to decide embedding dimensions if using embeddings.
Target: O3 is also a variable in the dataset. When creating windows, we need to ensure that future O3 values are not used as input. So windows must be constructed such that input features do not include future O3. That's standard.
For the global model: they want to train on all stations but predict per station. That means the model should learn from all stations, but we need to incorporate station information as a feature (categorical) to distinguish stations. That's fine. For per-station models, we train separate models for each station, so station is implicitly encoded.
Outlier handling: They need to define criteria for physical impossibility. For air quality variables (NO, NO2, NOx, O3), there are typical ranges. Wind direction (0-360), wind speed (>=0), temperature (reasonable), solar radiation (>=0). They need to provide domain knowledge or thresholds. We'll ask.
Imputation: They want per variable and per time range. For time series, imputation could be done using interpolation, forward fill, or more sophisticated methods. They need to specify which method for which variable. Possibly linear interpolation for short gaps, seasonal decomposition, etc. We'll ask.
Normalization: Different models require different scaling. Tree models don't need scaling, but neural networks do. So they need separate pipelines. They mention "normalización según modelo". So we need to handle that.
They want to create datasets per model (code 6) with two types: per station and all stations. That is essentially preparing the data in the right format for each model approach.
Model evaluation: They mention R^2, but for time series forecasting, other metrics like MAE, RMSE, etc. might be more appropriate. They can include multiple metrics.
They have not included a step for feature engineering beyond encoding. Maybe they want to add lag features or rolling statistics? But they have sliding windows, which will create sequences. For tree models, we might need to flatten the window into features. For RNN/LSTM, we keep the temporal dimension. So we need to handle differently.
Also, they haven't specified how to handle the multi-step output. For tree models, if using direct strategy, we'd create separate targets for each horizon. For RNN/LSTM, we can have multiple output units.
Given the complexity, we need to ask detailed questions to clarify:
Outlier definition: What are the physically impossible ranges for each variable? (e.g., NO < 0? O3 < 0? Wind direction 0-360? Wind speed >=0? Temperature range? Solar radiation >=0? Also, any known sensor errors?
Imputation methods: For each variable, what imputation method do you prefer? Options: linear interpolation, forward fill, backward fill, seasonal decomposition, or using other variables (regression imputation). Also, what is the maximum gap size to interpolate? Should we use time-based interpolation (e.g., hourly) or just fill all NaNs?
Forecasting horizon and window structure: You want to predict up to 72h. Do you want to predict the next 72 values (multi-step output) or just the next hour and then recursively? For tree models, do you want to create separate models for each forecast horizon or one model that outputs 72 values? For RNN/LSTM, do you want sequence-to-sequence (input sequence of past hours, output sequence of future hours)? What length of input window do you want to use? For example, use past 72 hours to predict next 72 hours?
Splitting strategy: For time series, we need to split chronologically without shuffling. How do you want to split? Typically train on earlier data, validate on later, test on the last part. For hyperparameter tuning, we can use time series cross-validation. Please specify the split ratios (e.g., 70% train, 15% val, 15% test) and if you want to use walk-forward validation.
Hyperparameter tuning: For each model, do you want to perform a grid search or random search? How many trials? Any specific hyperparameters to consider?
For the global model: How do you want to incorporate station information? One-hot encoding or embedding? If embedding, what dimension?
For categorical encoding: Do you want to use OneHotEncoder for tree models and embeddings for neural networks? Or a consistent method?
Normalization: For tree models, no scaling needed, but for neural networks, we need to scale features. Should we use MinMaxScaler or StandardScaler? Should we scale the target as well? Usually, for regression, scaling target can help neural networks. For multi-step, scaling is per variable.
Evaluation: What metrics do you want besides R^2? Possibly MAE, RMSE, and maybe visualization of predictions vs actuals.
Code structure: Do you want separate scripts for each code or one integrated pipeline? We'll provide code snippets that can be combined.
Now, we need to respond with a helpful analysis: confirm the steps, suggest any missing steps (like splitting before normalization, handling multi-step, etc.), and ask these clarifying questions in detail so that later we can produce precise code.

We should also note that the order might be: 1. Outlier, 2. Imputation, 3. Feature engineering (datetime cyclic, etc.), 4. Encoding categorical (but this depends on model, so maybe later), 5. Data splitting (chronological), 6. Normalization (fit on train only), 7. Window creation (but careful: windows should be created after splitting to avoid leakage? Actually, creating windows uses past values, so if we split first, we need to create windows within each set ensuring that windows don't cross the split boundary. Usually, we create windows from the entire dataset, then split the windows chronologically. That's acceptable because windows are based on past data only. But we must ensure that the target for a window does not include future data from beyond the split. So we can create windows first, then split the windows chronologically. However, if we normalize after splitting, we need to normalize each window's features based on training statistics. So the order could be: outlier/imputation, feature engineering, window creation, split, normalization. That works. We'll suggest that.

Also, for tree models, we might want to flatten the window into a feature vector. That step should be after window creation and before feeding to model. We can include that in the dataset creation step.

Now, craft the response.
El enfoque que planteas es sólido y cubre las etapas fundamentales para un proyecto de forecasting con múltiples modelos y estrategias. Sin embargo, para garantizar que los códigos sean precisos y evitar fugas de información (data leakage), es necesario ajustar el orden de algunos pasos y definir con más detalle ciertos aspectos. A continuación, te doy mi opinión sobre la secuencia propuesta y los puntos que convendría aclarar antes de generar los códigos.

1. Evaluación del orden de los pasos

Tu lista actual:

Outliers
Imputación
Codificación (datetime, dirección, ángulo, categóricas)
Normalización
Ventanas deslizantes
Creación de datasets (por estación y global)
7–10. Modelos (RF, XGBoost, RNN, LSTM) con partición, búsqueda de hiperparámetros, entrenamiento y validación
Problema potencial: La normalización (paso 4) se realiza antes de la partición de los datos. Esto puede causar data leakage si se calculan estadísticos (media, desviación, mínimos, máximos) usando todo el dataset, incluyendo el futuro. La solución es posponer la normalización después de la partición (o dentro de la validación cruzada). Además, para modelos de árbol (RF, XGBoost) la normalización no es necesaria, mientras que para redes neuronales (RNN, LSTM) sí lo es. Por tanto, deberás aplicar la normalización de forma condicional según el modelo.

Sugerencia de orden revisado:

Tratamiento de outliers (sin usar información futura, solo criterios físicos).
Imputación de valores NaN (usando métodos temporales que no miren hacia adelante, como interpolación lineal o forward fill).
Creación de variables temporales cíclicas (hora, día, mes, etc.) y transformación de ángulo/dirección a seno/coseno.
Codificación de variables categóricas (Estación, Transecto) según el modelo (OneHot para árboles, embeddings para redes).
Creación de ventanas deslizantes (secuencias de entrada y salida) a partir de la serie completa, pero manteniendo la cronología.
Partición de las ventanas en train/validation/test de forma cronológica (por ejemplo, 70% train, 15% validation, 15% test).
Normalización (solo para modelos que la requieran): se ajusta con los datos de train y se transforman train/validation/test.
Preparación de datasets específicos por modelo:

Para cada estación (modelos individuales)
Para el modelo global (incluyendo la estación como feature)
Entrenamiento, búsqueda de hiperparámetros y evaluación para cada modelo (RF, XGBoost, RNN, LSTM).
Nota: La codificación de categóricas (paso 4) también podría hacerse después de la partición para evitar leakage (por ejemplo, si se hace OneHot, no hay problema porque es una transformación determinista; pero si se usan embeddings, estos se aprenden durante el entrenamiento y no deben ver el test). En cualquier caso, es más seguro hacerlo después de la partición también. Sin embargo, como la codificación no usa información de la variable objetivo, se puede hacer antes con cuidado. Lo dejaremos como paso 4 antes de la partición, pero asegurándonos de que no se use información futura (lo cual no ocurre con codificación simple).

2. Preguntas clave para detallar los códigos

Para que los códigos se ajusten exactamente a tu problema, necesito que me aclares los siguientes puntos. Con esta información podré generar scripts precisos y listos para ejecutar.

a) Definición de outliers (físicamente imposibles)

¿Qué rangos consideras físicamente posibles para cada variable numérica? Por ejemplo:

NO, NO2, NOx, O3: ¿solo valores no negativos? ¿Hay algún límite superior razonable (ej. 500 µg/m³)?
Velocidad del viento: ¿mínimo 0, máximo razonable (ej. 20 m/s)?
Dirección: ¿siempre entre 0 y 360°? ¿Se permiten valores fuera de ese rango?
Temperatura: ¿rango típico de la zona (ej. -20°C a 45°C)?
Radiación solar: ¿no negativa, máximo (ej. 1200 W/m²)?
Distancia y ángulo: ¿siempre positivos? ¿Ángulo entre 0 y 360?
¿Hay algún código de error conocido (ej. -999, 9999) que deba tratarse como NaN?
b) Métodos de imputación

Para cada variable, ¿qué técnica prefieres? Opciones comunes:

Interpolación lineal (para huecos pequeños)
Forward fill / backward fill (para datos que cambian lentamente)
Media móvil (ventana de tiempo)
Imputación estacional (por hora del día, mes)
Imputación multivariante (usando otras variables, ej. regresión)
¿Cuál es el tamaño máximo de hueco que te gustaría interpolar? ¿O prefieres imputar todos los NaN independientemente de la longitud del gap?
¿Quieres que la imputación se haga por estación por separado o considerando todas las estaciones a la vez?
c) Horizonte de predicción y estructura de ventanas

Dices "predicción horaria hasta 72h". ¿Quieres predecir los próximos 72 valores de O3 (multi-step) o solo el siguiente valor y luego usar recursividad? Lo habitual en ML es multi-step directo (un modelo por horizonte o un modelo con 72 salidas) o secuencia a secuencia.
¿Qué longitud de secuencia de entrada (pasado) quieres usar? Por ejemplo, ¿usar las últimas 72 horas para predecir las siguientes 72? ¿O una ventana más corta (ej. 24h)?
Para los modelos de árbol (RF, XGBoost): ¿prefieres crear un modelo separado para cada horizonte (72 modelos) o un único modelo que prediga los 72 valores a la vez (requiere transformar la salida en una variable multirrespuesta)? En XGBoost se puede hacer multi-output con multioutput o usando sklearn.multioutput.MultiOutputRegressor.
Para RNN/LSTM: ¿quieres una arquitectura sequence-to-sequence (encoder-decoder) o simplemente muchos a muchos con una salida para cada paso futuro? Lo más simple es usar muchas a muchas con una capa densa final con 72 neuronas.
d) Estrategia de partición temporal

¿Qué porcentajes asignarías a train, validation y test? Ejemplo: 70% train, 15% validation, 15% test (manteniendo el orden cronológico).
Para la búsqueda de hiperparámetros, ¿quieres usar validación cruzada en series temporales (TimeSeriesSplit) o una simple partición fija train/validation?
¿Tienes alguna fecha de corte específica (ej. entrenar con datos hasta 2024 y testear con 2025)?
e) Hiperparámetros a optimizar

Para Random Forest: ¿qué rango de valores quieres probar (n_estimators, max_depth, min_samples_split, etc.)?
Para XGBoost: ¿parámetros como learning_rate, max_depth, subsample, colsample_bytree, n_estimators?
Para RNN/LSTM: ¿número de capas, unidades por capa, dropout, learning rate, batch size, épocas? ¿Usarás early stopping?
¿Número de iteraciones de búsqueda (grid search o random search)?
f) Modelo global: tratamiento de la estación

Para el modelo que entrena con todas las estaciones, ¿cómo quieres codificar la variable "Estación"? Las opciones son:

One-hot encoding (adecuado para árboles y redes con pocas estaciones).
Embedding (para redes, si hay muchas estaciones, pero con unas pocas no aporta mucho).
Simplemente ignorarla y esperar que el modelo aprenda diferencias mediante otras variables (no recomendable).
¿Y "Transecto"? ¿Es relevante o puedes ignorarlo? Parece constante en el ejemplo, pero quizá varíe entre estaciones.
g) Codificación de variables cíclicas

Además de dirección y ángulo, para datetime quieres extraer: día del año, mes, semana, hora. ¿Quieres convertirlas a seno/coseno también? Por ejemplo:

hour_sin = sin(2π * hour/24), hour_cos = cos(2π * hour/24)
month_sin = sin(2π * month/12), etc.
¿Quieres incluir también otras características como día de la semana (0-6) con one-hot o también seno/coseno? Lo típico es one-hot para día de semana (no es periódico en el sentido circular, pero se puede tratar como categoría).
h) Normalización (escalado)

Para redes neuronales: ¿Prefieres MinMaxScaler (escala a [0,1]) o StandardScaler (media 0, std 1)? Normalmente MinMax va bien para datos con límites conocidos; Standard para distribuciones Gaussianas.
¿Quieres escalar también la variable objetivo (O3)? Si escalas el target, las predicciones saldrán escaladas y deberás invertir la transformación para calcular métricas.
¿Debemos aplicar la misma normalización a todas las características numéricas, incluyendo las transformadas seno/coseno (que ya están en [-1,1])? Generalmente no es necesario reescalarlas, pero no perjudica.
i) Métricas de evaluación

Además de R², ¿quieres calcular MAE, RMSE, MAPE (si no hay ceros)? ¿Alguna métrica específica para la calidad del forecasting (ej. Índice de acuerdo de Willmott)?
¿Quieres visualizar las predicciones vs reales para algunos horizontes (ej. horizonte 1, 24, 48, 72) en una gráfica por estación?
j) Entorno y librerías

¿Tienes preferencia por alguna librería específica para deep learning (TensorFlow/Keras, PyTorch)? Usaré Keras por simplicidad a menos que indiques otra.
¿Los datos están en un único CSV o en archivos separados por estación? En el ejemplo parece un único DataFrame con columna "Estacion". Asumiré un único CSV.
3. Pasos adicionales a considerar

Análisis exploratorio inicial: Aunque no lo has pedido, sería recomendable un código 0 para visualizar la distribución de datos, estacionalidades, correlaciones, etc., pero no es imprescindible para los modelos.
Feature engineering adicional: Podrías añadir medias móviles de contaminantes, variables de retardo (lags) específicas, pero las ventanas deslizantes ya capturan dependencias temporales. Si quieres incluir lags explícitos para árboles, podríamos aplanar la ventana.
Guardar modelos y resultados: Quizá quieras guardar los modelos entrenados y las métricas en un archivo.
4. Conclusión

El plan es coherente, pero requiere los ajustes mencionados en el orden y una definición más precisa de los detalles. Una vez que me respondas a las preguntas anteriores, podré generarte los 10 códigos (o los que finalmente sean) con explicaciones y adaptados a tus necesidades. Quedo a la espera de tu respuesta para continuar.
a) 1. Para NO, NO2, NOx, O3 valores no negativos < 800 µg/m³
2. Para velocidad de viento (Veloc.) mínimo 0 máximo 60m/s
3.para direccional de viento (Direc.) y Angulo siempre entre 0 y 260 ya que están en grados
4. para temperatura de -40 a 50
5. para radiación solar no negativa y máximo 1200
6. Distancia siempre positiva y Angulo entre 0 y 360
No hay ningún código de error conocido

b) 1. Quiero que el método de imputación de cada variable sea el mejor. Para ello quiero que se divida la imputación por escala temporal de datos faltantes, es decir si el rango de datos faltantes en menor o igual a 6h quiero que sea una imputación por interpolación temporal o lineal, si es mayor a 6h y menor de 72h quiero que sea una imputación estaciónale por hora del día y si los datos faltantes son mayor a 72h quiero que se haga una imputación multivariante. Aunque me gustaría que reflexionases y me dijeses cual es mejor para cada caso.

2. Quiero que se haga una imputación por estación para el método de por estaciónes y otra imputación por métodos de todas las estaciones.

c) 1. Quiero una predicción multi-step
2. quiero usar las últimas 72 horas para predecir las siguientes 72
3. prefieroun único modelo que prediga los 72 valores a la vez
4. quiero una arquitectura sequence-to-sequence (encoder-decoder)

d) 1. quiero 70% train, 15% validation, 15% test (manteniendo el orden cronológico)
2. quiero usar validación cruzada en series temporales (TimeSeriesSplit)
3. quiero un trais desde el inicio de los datos hasta 2023-12-31, un validación de 2024-1-1 a 2024-12-31 y un test de 2025-1-1 a 2025-12-31

e) 1. Quiero probar muchos hiperprametros con grid search

f) 1. para la variable estación haz onehotencodding para ML y embedding para DL
2. para la variable transecto lo mismo

g)Si, quiero codificar la hora, el día, la semana, el mes y el año en seno y coseno. No quiero onehotencoder para la variable datetime

h) 1. Para redes neuronales quiero MinMaxScaler
2. si es posible y coherente no escalar O3 prefiero que no se escale, pero si no es coherente no escalarlo entonces prefiero que se escale.
3. Aplica la normalización a todas las características numéricas incluyendo seno y coseno

i) 1. Quiero que se calculen todas las métricas mas comunes y también el  Índice de acuerdo de Willmott
2. si, quiero visualizar las predicciones vs reales para algunos horizontes (24, 48 y 72h)

j) 1. Me da igual la libreria que se utilize.
2. Tengo un csv por estación:

station_paths = [
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E1_Alicante.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E1_Elche.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E2_Buñol.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E2_Villar_Arzobispo.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E1_Castellon.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E2_Onda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E2_Coratxa.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E3_Zorita.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E2_Morella.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E3_Zorita.csv",
]


Quiero guardar las métricas y los modelos en un archivo.









Dada esta información y el siguiente código, creame el primer código COMPLETO listo para copiar, pegar y ejecutar.




Código ejemplo para primer código (basta en el pero que no sea igual):

# Código 1: Detección y clasificación de outliers (sensor_error vs evento_real)
# Diseñado para ejecutarse en un notebook (VSCode, Python 3.11).
# Dependencias: pandas, numpy, scikit-learn, statsmodels
# Instalar si es necesario: pip install pandas numpy scikit-learn statsmodels

import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from statsmodels.tsa.seasonal import STL

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# ---------------------------
# Parámetros configurables
# ---------------------------
PERIOD_H = 24                      # periodo estacional (horario → 24h)
STL_K_MAD = 4.0                    # umbral en MAD para residuo STL
JUMP_K_MAD = 6.0                   # umbral para saltos 1h basados en MAD local
ISO_CONTAMINATION = 0.01           # proporción para IsolationForest
ROLL_WINDOW_H = 24                 # ventana (horas) para calcular MAD local
PHYS_TEMP_MIN = -50.0
PHYS_TEMP_MAX = 60.0
PHYS_RSOL_MIN = 0.0
PHYS_WIND_MIN = 0.0
ANGLE_MIN = 0.0
ANGLE_MAX = 360.0
OUTPUT_DIR = os.path.expanduser("~/Documents/GitHub/TFGFinal/Datos_TFG_outliers")  # carpeta salida
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------------------
# Lista de rutas (usa la tuya)
# ---------------------------
station_paths = [
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E1_Alicante.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E1_Elche.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E2_Buñol.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E2_Villar_Arzobispo.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E1_Castellon.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E2_Onda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E2_Coratxa.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E3_Zorita.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E2_Morella.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E3_Zorita.csv",
]
station_paths = [os.path.expanduser(p) for p in station_paths]

# ---------------------------
# Funciones auxiliares
# ---------------------------
def robust_mad(x):
    """MAD: median absolute deviation (no-Bessel)."""
    x = np.asarray(x)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    return mad if mad > 0 else np.nanmean(np.abs(x - med))  # fallback

def ensure_datetime_index(df, datetime_col=None, freq='h'):
    """Asegura índice datetime con frecuencia horaria y reindexa dejando NaNs donde faltan timestamps."""
    if datetime_col is None:
        # intentar encontrar columna timestamp
        candidates = [c for c in df.columns if c.lower() in ("datetime","fecha","time","timestamp")]
        if not candidates:
            raise ValueError("No se encontró columna datetime. Pasa el nombre con datetime_col.")
        datetime_col = candidates[0]
    df = df.copy()
    df[datetime_col] = pd.to_datetime(df[datetime_col], errors='coerce')
    df = df.set_index(datetime_col).sort_index()
    # rellenar para rango completo
    full_idx = pd.date_range(start=df.index.min(), end=df.index.max(), freq='h')
    df = df.reindex(full_idx)
    df.index.name = 'datetime'
    return df


def physical_checks(df):
    """Marca errores físicos obvios."""
    flags = pd.DataFrame(index=df.index)
    flags['physical_error'] = False
    flags['reason'] = ""
    # R.Sol. < 0
    if 'R.Sol.' in df.columns:
        mask = (df['R.Sol.'] < PHYS_RSOL_MIN) & (~df['R.Sol.'].isna())
        flags.loc[mask, 'physical_error'] = True
        flags.loc[mask, 'reason'] = flags.loc[mask, 'reason'] + "R.Sol. < 0; "
    # Veloc < 0
    if 'Veloc.' in df.columns:
        mask = (df['Veloc.'] < PHYS_WIND_MIN) & (~df['Veloc.'].isna())
        flags.loc[mask, 'physical_error'] = True
        flags.loc[mask, 'reason'] = flags.loc[mask, 'reason'] + "Veloc < 0; "
    # Angulo/Direc fuera de rango
    for col in ['Direc.','Angulo(en grados)','Angle','Dirección']:
        if col in df.columns:
            mask = (~df[col].isna()) & ((df[col] < ANGLE_MIN) | (df[col] >= ANGLE_MAX))
            flags.loc[mask, 'physical_error'] = True
            flags.loc[mask, 'reason'] = flags.loc[mask, 'reason'] + f"{col} fuera [0,360); "
    # Temp extrema
    if 'Temp' in df.columns:
        mask = (~df['Temp'].isna()) & ((df['Temp'] < PHYS_TEMP_MIN) | (df['Temp'] > PHYS_TEMP_MAX))
        flags.loc[mask, 'physical_error'] = True
        flags.loc[mask, 'reason'] = flags.loc[mask, 'reason'] + "Temp extrema; "
    # O3 negativa
    for o3_col in ['O3','O₃','Ozone']:
        if o3_col in df.columns:
            mask = (~df[o3_col].isna()) & (df[o3_col] < 0)
            flags.loc[mask, 'physical_error'] = True
            flags.loc[mask, 'reason'] = flags.loc[mask, 'reason'] + f"{o3_col} < 0; "
    return flags

def stl_outliers(df, col='O3', period=PERIOD_H, k=STL_K_MAD):
    """Detecta outliers en la serie usando STL residuo + MAD"""
    res = pd.Series(False, index=df.index)
    try:
        s = df[col].dropna()
        if len(s) > period*2:
            stl = STL(s, period=period, robust=True)
            out = stl.fit()
            resid = out.resid
            mad_resid = robust_mad(resid)
            threshold = k * mad_resid
            mask = np.abs(resid) > threshold
            res.loc[mask.index] = mask.values
    except Exception as e:
        # si falla, devolvemos serie False
        print(f"STL failed for {col}: {e}")
    return res

def rolling_jump_outliers(df, col='O3', window=ROLL_WINDOW_H, k=JUMP_K_MAD):
    """Detecta saltos bruscos 1h comparando diffs y MAD local."""
    res = pd.Series(False, index=df.index)
    s = df[col]
    # diff 1h (forward/backward)
    diff = s.diff().abs()
    # calc rolling MAD of diffs
    roll_mad = diff.rolling(window=window, min_periods=1, center=True).apply(lambda x: robust_mad(x), raw=True)
    thr = k * roll_mad
    mask = (diff > thr) & (~diff.isna())
    res.loc[mask.index] = mask.values
    return res

def isolation_forest_outliers(df, features=None, contamination=ISO_CONTAMINATION):
    """IsolationForest multivariado (retorna boolean series)."""
    if features is None:
        candidates = ['O3','NO','NO2','NOx','Temp','Veloc.','R.Sol.']
        features = [c for c in candidates if c in df.columns]
    res = pd.Series(False, index=df.index)
    sub = df[features].copy()
    valid_idx = sub.dropna(how='all').index
    if len(valid_idx) < 20:
        return res
    sub = sub.loc[valid_idx]
    try:
        # Rellenamos hacia adelante, hacia atrás y finalmente ceros si aún quedan NaN
        sub = sub.ffill().bfill().fillna(0)
        iso = IsolationForest(contamination=contamination, random_state=RANDOM_STATE)
        pred = iso.fit_predict(sub.values)
        out_idx = sub.index[pred == -1]
        res.loc[out_idx] = True
    except Exception as e:
        print("IsolationForest failed:", e)
    return res

def neighbor_support(df, idx, col='O3', window=3, rel_tol=0.5):
    """
    Evalúa si un pico en idx tiene soporte en vecinos (vecinos muestran cambios coherentes).
    window: número total de puntos a mirar (debe ser impar), default 3 -> t-1, t, t+1
    rel_tol: fracción del tamaño del cambio que consideraremos 'soporte'
    """
    half = (window - 1) // 2
    i_pos = df.index.get_loc(idx)
    start = max(0, i_pos - half)
    end = min(len(df) - 1, i_pos + half)
    vals = df[col].iloc[start:end+1].values
    center = df[col].iloc[i_pos]
    # si la mayoría de vecinos están NaN → no hay soporte
    if np.isnan(vals).sum() >= half + 1:
        return False
    # support heuristic: si cambio del centro respecto a median vecino es mayor que
    # median(neighbors) * rel_tol -> we consider neighbors support if they change similarly
    med_neighbors = np.nanmedian(np.concatenate([vals[:half], vals[half+1:]])) if len(vals) > 1 else np.nan
    if np.isnan(med_neighbors):
        return False
    # if both neighbors changed in same direction as center relative to older values -> support
    # Simpler: if abs(center - med_neighbors) < abs(center) * rel_tol => we say has support
    if abs(center - med_neighbors) <= abs(center) * rel_tol:
        return True
    return False

# ---------------------------
# Pipeline para una estación
# ---------------------------
def process_station(path, save_out=True, verbose=True):
    """Carga CSV de estación, detecta outliers y clasifica sensor_error/evento_real/doubtful.
       Devuelve dataframe con columnas nuevas y guarda CSV con sufijo _outliers.csv"""
    station_name = Path(path).stem
    if verbose:
        print("Procesando:", station_name)
    df_raw = pd.read_csv(path)
    df = ensure_datetime_index(df_raw, datetime_col=None, freq='h')
    # Inicializar flags
    flags = pd.DataFrame(index=df.index)
    flags['physical_error'] = False
    flags['physical_reason'] = ""
    # physical checks
    phys = physical_checks(df)
    flags['physical_error'] = phys['physical_error']
    flags['physical_reason'] = phys['reason']

    # detectar outliers univariados por STL (O3 si existe)
    o3_col = next((c for c in df.columns if c in ['O3']), None)
    if o3_col is None:
        print(f"Warning: {station_name} no tiene columna O3 reconocida. Saltando STL/jump para O3.")
        stl_mask = pd.Series(False, index=df.index)
        jump_mask = pd.Series(False, index=df.index)
    else:
        stl_mask = stl_outliers(df, col=o3_col, period=PERIOD_H, k=STL_K_MAD)
        jump_mask = rolling_jump_outliers(df, col=o3_col, window=ROLL_WINDOW_H, k=JUMP_K_MAD)

    flags['outlier_stl'] = stl_mask
    flags['outlier_jump'] = jump_mask

    # Isolation Forest multivariado
    iso_mask = isolation_forest_outliers(df, features=None, contamination=ISO_CONTAMINATION)
    flags['outlier_iso'] = iso_mask

    # Clasificación heurística sensor_error vs evento_real vs doubtful
    flags['sensor_error'] = False
    flags['evento_real'] = False
    flags['doubtful'] = False

    # reglas básicas
    for ts in df.index:
        reasons = []
        # physical error overrides
        if flags.at[ts, 'physical_error']:
            flags.at[ts, 'sensor_error'] = True
            reasons.append("physical")
            continue
        # detector combinados
        is_stl = bool(flags.at[ts, 'outlier_stl'])
        is_jump = bool(flags.at[ts, 'outlier_jump'])
        is_iso = bool(flags.at[ts, 'outlier_iso'])
        any_det = is_stl or is_jump or is_iso
        if not any_det:
            # no outlier -> nothing to do
            continue
        # if multiple detectors agree → more fiable
        n_det = sum([is_stl, is_jump, is_iso])
        # check neighbour support only for O3
        support = False
        if o3_col is not None and not np.isnan(df.at[ts,o3_col]):
            try:
                support = neighbor_support(df, ts, col=o3_col, window=3, rel_tol=0.6)
            except Exception:
                support = False
        # Heurística:
        # - Si hay detección pero sin soporte en vecinos y detección es 'aislada' -> probable sensor_error
        # - Si varios detectores y hay soporte / hay picos también en NO/NO2/NOx/Temp/R.Sol -> evento_real
        related_vars = ['NO','NO2','NOx','Temp','R.Sol.','Veloc.']
        related_present = any([(v in df.columns) and (not pd.isna(df.at[ts,v])) for v in related_vars])
        # check if related vars also spike (simple: compare to rolling median)
        related_spike = False
        for v in related_vars:
            if v in df.columns:
                val = df.at[ts,v]
                if pd.isna(val):
                    continue
                rolling_med = df[v].rolling(window=ROLL_WINDOW_H, min_periods=1, center=True).median().iloc[df.index.get_loc(ts)]
                if not pd.isna(rolling_med):
                    if abs(val - rolling_med) > 2 * robust_mad(df[v].dropna()):  # simple threshold
                        related_spike = True
                        break
        # classification rules:
        if n_det >= 2 and related_spike:
            flags.at[ts, 'evento_real'] = True
            reasons.append("multi_det + related_spike")
        else:
            # if isolated detection and no neighbor support => sensor error
            if not support and n_det >= 1:
                flags.at[ts, 'sensor_error'] = True
                reasons.append("isolated_no_support")
            else:
                # ambiguous: iso true but no clear support or multiple detectors but no related spike
                flags.at[ts, 'doubtful'] = True
                reasons.append("ambiguous")
        # end loop

    # Crear columna de imputation: si sensor_error True -> poner NaN (preparar imputación)
    df_proc = df.copy()
    if o3_col is not None:
        df_proc['O3_for_impute'] = df_proc[o3_col].where(~flags['sensor_error'], np.nan)

    # adjuntar flags al df final
    out_df = pd.concat([df_proc, flags], axis=1)

    # guardar CSV con sufijo
    if save_out:
        out_path = os.path.join(OUTPUT_DIR, f"{station_name}_outliers.csv")
        out_df.to_csv(out_path, index=True)
        if verbose:
            print(f"Guardado: {out_path}")
    return out_df

# ---------------------------
# Procesar todas las estaciones y compilar doubtfuls
# ---------------------------
all_doubtful = []
all_summaries = []
for p in station_paths:
    try:
        df_out = process_station(p, save_out=True, verbose=True)
        # extraer filas dudosas
        if 'doubtful' in df_out.columns:
            doubtful_rows = df_out[df_out['doubtful'] == True].copy()
            if not doubtful_rows.empty:
                # añadir columna estación
                doubtful_rows['station_file'] = Path(p).stem
                all_doubtful.append(doubtful_rows)
        # summary counts
        summary = {
            'station': Path(p).stem,
            'n_rows': len(df_out),
            'n_physical_error': int(df_out['physical_error'].sum()) if 'physical_error' in df_out.columns else 0,
            'n_outlier_stl': int(df_out['outlier_stl'].sum()) if 'outlier_stl' in df_out.columns else 0,
            'n_outlier_jump': int(df_out['outlier_jump'].sum()) if 'outlier_jump' in df_out.columns else 0,
            'n_outlier_iso': int(df_out['outlier_iso'].sum()) if 'outlier_iso' in df_out.columns else 0,
            'n_sensor_error': int(df_out['sensor_error'].sum()) if 'sensor_error' in df_out.columns else 0,
            'n_evento_real': int(df_out['evento_real'].sum()) if 'evento_real' in df_out.columns else 0,
            'n_doubtful': int(df_out['doubtful'].sum()) if 'doubtful' in df_out.columns else 0,
        }
        all_summaries.append(summary)
    except Exception as e:
        print(f"Error procesando {p}: {e}")

# guardar doubtfuls combinados
if all_doubtful:
    combined = pd.concat(all_doubtful, axis=0)
    combined_path = os.path.join(OUTPUT_DIR, "doubtful_cases_combined.csv")
    combined.to_csv(combined_path, index=True)
    print(f"CSV combinado de casos dudosos guardado en: {combined_path}")
# guardar resumen
summary_df = pd.DataFrame(all_summaries)
summary_path = os.path.join(OUTPUT_DIR, "outliers_summary.csv")
summary_df.to_csv(summary_path, index=False)
print("Resumen guardado en:", summary_path)

print("Proceso completado. Revisa la carpeta de salida para CSVs por estación, summary y casos dudosos.")
El usuario ha dado respuestas detalladas a las preguntas y ha proporcionado un código de ejemplo para el primer paso (outliers). Ahora pide que, dada esa información y el código de ejemplo, le cree el primer código COMPLETO listo para copiar, pegar y ejecutar.

Instrucciones: "Dada esta información y el siguiente código, creame el primer código COMPLETO listo para copiar, pegar y ejecutar." El "siguiente código" es el código de ejemplo que él ha incluido. Por tanto, debemos generar el Código 1: "Conservación o eliminación de outliers según si son físicamente imposibles y errores de medición, dato corrupto (eliminar) o si son eventos extremos reales (conservar)."

El usuario ha dado especificaciones en su respuesta anterior (a, b, c, ...). Para el código 1, necesitamos adaptar el código de ejemplo a sus especificaciones concretas:

Rangos físicos: NO, NO2, NOx, O3 < 800 µg/m³, no negativos; Veloc. 0-60 m/s; Direc. y Angulo 0-360; Temp -40 a 50; R.Sol. 0-1200; Distancia positiva. En el ejemplo, hay rangos definidos (PHYS_TEMP_MIN, etc.) pero no coinciden exactamente. Debemos ajustarlos.
No hay códigos de error conocidos.
El código de ejemplo clasifica outliers en sensor_error, evento_real, doubtful usando varios métodos (STL, jump, IsolationForest). El usuario no ha especificado métodos concretos para outliers, pero en su descripción inicial decía "según si son físicamente imposibles y errores de medición, dato corrupto (eliminar) o si son eventos extremos reales (conservar)". El código de ejemplo ya implementa una lógica de clasificación, lo cual parece adecuado. Podemos mantener la estructura pero ajustar umbrales y parámetros según sus rangos físicos. Además, podemos simplificar o adaptar a sus necesidades.
El usuario no ha dado criterios específicos para detectar outliers más allá de los rangos físicos. El código de ejemplo usa STL, jump y IsolationForest. Eso está bien para una primera aproximación. Pero debemos asegurarnos de que la clasificación final (sensor_error vs evento_real) se base en las reglas que él pueda aprobar. Podemos mantener la lógica del ejemplo, que es heurística, pero documentarla bien.

También el usuario quiere que se guarden los resultados (los CSVs con flags). El código de ejemplo ya lo hace.

Entonces, para el Código 1, podemos usar el código de ejemplo como base, pero modificando:

Los límites físicos según sus especificaciones.
Ajustar nombres de columnas (Direc., Angulo, etc.) para que coincidan con su dataset.
Posiblemente, simplificar la parte de neighbor_support para que funcione con cualquier variable, no solo O3. Pero en el ejemplo solo se aplica a O3. Podemos dejarlo así.
Añadir la variable objetivo O3 como referencia, pero el código de ejemplo ya la usa.
Además, en su respuesta, dijo: "Quiero guardar las métricas y los modelos en un archivo." Eso es para códigos posteriores. Para el código 1 solo guardamos los datos con outliers marcados.

Por tanto, procedamos a generar el código 1 completo, con explicaciones y adaptado a sus rutas, rangos, y con la misma funcionalidad que el ejemplo pero personalizado.

Debemos incluir en el código las rutas de los archivos que él proporcionó. Ya las ha dado en station_paths. También la carpeta de salida: puede ser la misma que en el ejemplo (OUTPUT_DIR) pero quizás conviene crear una subcarpeta específica para outliers.

También hay que tener en cuenta que el código de ejemplo define ensure_datetime_index que asume frecuencia horaria y reindexa. Eso es correcto.

Posibles mejoras: en la clasificación, el ejemplo usa neighbor_support solo para O3, y luego related_spike para otras variables. Podemos mantenerlo.

Asegurémonos de que los nombres de columnas coinciden con los del dataset: según el ejemplo inicial, las columnas son: datetime, NO, NO2, NOx, O3, Veloc., Direc., Temp., R.Sol., Estacion, Transecto, Dist., Angulo. El código de ejemplo usa 'R.Sol.', 'Veloc.', 'Direc.', 'Temp', 'Angulo(en grados)'? En el ejemplo, para ángulo busca 'Direc.','Angulo(en grados)','Angle','Dirección'. Nosotros debemos usar 'Direc.' y 'Angulo'. Ajustaremos.

También la variable 'Dist.' aparece como 'Dist.' en el dataset, no la hemos usado en outlier detection, pero podemos dejarla fuera.

El código de ejemplo también tiene 'related_vars' que incluye 'NO','NO2','NOx','Temp','R.Sol.','Veloc.'. Eso está bien.

Vamos a producir el código con comentarios en español, ajustando los parámetros físicos:

PHYS_TEMP_MIN = -40, PHYS_TEMP_MAX = 50
PHYS_RSOL_MIN = 0, PHYS_RSOL_MAX = 1200 (aunque en el ejemplo no usa max, solo min)
PHYS_WIND_MIN = 0, PHYS_WIND_MAX = 60
ANGLE_MIN = 0, ANGLE_MAX = 360
Para NO, NO2, NOx, O3: rango 0 a 800. En el ejemplo no hay un chequeo físico específico para estos contaminantes aparte de O3 < 0. Deberíamos añadir un chequeo para valores > 800. En la función physical_checks podemos añadir condiciones para NO, NO2, NOx y O3 que verifiquen que no excedan 800.
También, en el ejemplo, para O3 ya comprueba <0. Extenderemos a >800.

También para Veloc. >60.

Ajustemos la función physical_checks.

En el código de ejemplo, la función physical_checks devuelve flags con 'physical_error' y 'reason'. Añadiremos las nuevas comprobaciones.

Además, en la parte de clasificación, las reglas deciden sensor_error si es físico imposible, o si es aislado sin soporte. Eso puede estar bien. Pero como el usuario no ha dado reglas específicas, mantendremos la heurística del ejemplo.

También, en el ejemplo, se crea una columna 'O3_for_impute' donde se pone NaN si sensor_error es True. Eso es para preparar la imputación (siguiente código). Está bien.

Ahora, generemos el código. Debe ser autónomo, con todas las funciones y el bucle de procesamiento. Pondré comentarios para que el usuario entienda.

Además, añadiré al final una impresión de resumen y guardado de los casos dudosos y resumen.

El código usará las rutas de estación proporcionadas, expandiendo el ~.

Vamos a ello.
He ajustado el código de ejemplo a tus especificaciones concretas: rangos físicos (NO, NO2, NOx, O3 < 800 µg/m³, velocidad hasta 60 m/s, temperatura -40 a 50 °C, radiación 0‑1200, ángulos 0‑360), nombres de columnas exactos y la misma filosofía de clasificación (sensor error vs evento real). El código está listo para copiar, pegar y ejecutar en un entorno con Python 3.11 y las librerías indicadas. Generará una carpeta outliers dentro de tu directorio de trabajo, guardará un CSV por estación con las marcas de outliers y un resumen global.

python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 1: Detección y clasificación de outliers (error de sensor vs evento real)
para datos horarios de calidad del aire.

Ejecutar en VSCode / Jupyter / terminal con Python 3.11.
Dependencias: pandas, numpy, scikit-learn, statsmodels
Instalación rápida: pip install pandas numpy scikit-learn statsmodels
"""

import os
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from statsmodels.tsa.seasonal import STL

warnings.filterwarnings("ignore")
RANDOM_STATE = 42

# -------------------------------------------------------------------
# PARÁMETROS CONFIGURABLES (ajustados a tus especificaciones)
# -------------------------------------------------------------------
PERIOD_H = 24                      # periodo estacional (horario → 24h)
STL_K_MAD = 4.0                    # umbral en MAD para residuo STL
JUMP_K_MAD = 6.0                   # umbral para saltos 1h basados en MAD local
ISO_CONTAMINATION = 0.01           # proporción esperada de outliers para IsolationForest
ROLL_WINDOW_H = 24                 # ventana (horas) para calcular MAD local

# Límites físicos (según tus respuestas)
PHYS_TEMP_MIN = -40.0
PHYS_TEMP_MAX = 50.0
PHYS_RSOL_MIN = 0.0
PHYS_RSOL_MAX = 1200.0
PHYS_WIND_MIN = 0.0
PHYS_WIND_MAX = 60.0
ANGLE_MIN = 0.0
ANGLE_MAX = 360.0
CONTAMINANTE_MAX = 800.0           # para NO, NO2, NOx, O3 (µg/m³)

# Carpeta de salida (se crea si no existe)
OUTPUT_DIR = os.path.expanduser("~/Documents/GitHub/TFGFinal/Datos_TFG_outliers")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -------------------------------------------------------------------
# RUTAS DE LAS ESTACIONES (tal como las proporcionaste)
# -------------------------------------------------------------------
station_paths = [
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E1_Alicante.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T1_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E1_Elche.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T2_E2_Elda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T3_E2_Buñol.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E1_Valencia.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T4_E2_Villar_Arzobispo.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E1_Castellon.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T6_E2_Onda.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E2_Coratxa.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T7_E3_Zorita.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E1_Sant_Jordi.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E2_Morella.csv",
    r"~/Documents/GitHub/TFGFinal/Datos TFG/DATOSFINALES/HORARIOS/T8_E3_Zorita.csv",
]
# Expandir '~' a la ruta completa del usuario
station_paths = [os.path.expanduser(p) for p in station_paths]

# -------------------------------------------------------------------
# FUNCIONES AUXILIARES
# -------------------------------------------------------------------
def robust_mad(x):
    """MAD: median absolute deviation (robusto)."""
    x = np.asarray(x)
    med = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - med))
    return mad if mad > 0 else np.nanmean(np.abs(x - med))  # fallback

def ensure_datetime_index(df, datetime_col=None, freq='h'):
    """
    Convierte la columna datetime a índice DatetimeIndex, reindexa a frecuencia horaria
    (dejando NaN en los huecos) y ordena.
    """
    if datetime_col is None:
        # Buscar columna típica de fecha
        candidates = [c for c in df.columns if c.lower() in ("datetime", "fecha", "time", "timestamp")]
        if not candidates:
            raise ValueError("No se encontró columna datetime. Especifica datetime_col.")
        datetime_col = candidates[0]
    df = df.copy()
    df[datetime_col] = pd.to_datetime(df[datetime_col], errors='coerce')
    df = df.set_index(datetime_col).sort_index()
    # Rellenar para que no falten horas
    full_idx = pd.date_range(start=df.index.min(), end=df.index.max(), freq='h')
    df = df.reindex(full_idx)
    df.index.name = 'datetime'
    return df

def physical_checks(df):
    """
    Marca como error físico aquellos valores fuera de los rangos definidos.
    Devuelve DataFrame con columnas 'physical_error' (bool) y 'physical_reason' (str).
    """
    flags = pd.DataFrame(index=df.index)
    flags['physical_error'] = False
    flags['physical_reason'] = ""

    # Radiación solar
    if 'R.Sol.' in df.columns:
        mask = (df['R.Sol.'] < PHYS_RSOL_MIN) | (df['R.Sol.'] > PHYS_RSOL_MAX) & (~df['R.Sol.'].isna())
        flags.loc[mask, 'physical_error'] = True
        flags.loc[mask, 'physical_reason'] += "R.Sol. fuera [0,1200]; "

    # Velocidad viento
    if 'Veloc.' in df.columns:
        mask = (df['Veloc.'] < PHYS_WIND_MIN) | (df['Veloc.'] > PHYS_WIND_MAX) & (~df['Veloc.'].isna())
        flags.loc[mask, 'physical_error'] = True
        flags.loc[mask, 'physical_reason'] += "Veloc. fuera [0,60]; "

    # Dirección y Ángulo (0-360)
    for col in ['Direc.', 'Angulo']:
        if col in df.columns:
            mask = (~df[col].isna()) & ((df[col] < ANGLE_MIN) | (df[col] >= ANGLE_MAX))
            flags.loc[mask, 'physical_error'] = True
            flags.loc[mask, 'physical_reason'] += f"{col} fuera [0,360); "

    # Temperatura
    if 'Temp.' in df.columns:
        mask = (~df['Temp.'].isna()) & ((df['Temp.'] < PHYS_TEMP_MIN) | (df['Temp.'] > PHYS_TEMP_MAX))
        flags.loc[mask, 'physical_error'] = True
        flags.loc[mask, 'physical_reason'] += "Temp. fuera [-40,50]; "

    # Contaminantes (NO, NO2, NOx, O3): no negativos y < 800
    for pol in ['NO', 'NO2', 'NOx', 'O3']:
        if pol in df.columns:
            mask = (~df[pol].isna()) & ((df[pol] < 0) | (df[pol] > CONTAMINANTE_MAX))
            flags.loc[mask, 'physical_error'] = True
            flags.loc[mask, 'physical_reason'] += f"{pol} fuera [0,800]; "

    # Distancia (positiva)
    if 'Dist.' in df.columns:
        mask = (~df['Dist.'].isna()) & (df['Dist.'] < 0)
        flags.loc[mask, 'physical_error'] = True
        flags.loc[mask, 'physical_reason'] += "Dist. negativa; "

    return flags

def stl_outliers(df, col='O3', period=PERIOD_H, k=STL_K_MAD):
    """Detecta outliers mediante residuos STL + umbral MAD."""
    res = pd.Series(False, index=df.index)
    try:
        s = df[col].dropna()
        if len(s) > period * 2:
            stl = STL(s, period=period, robust=True)
            out = stl.fit()
            resid = out.resid
            mad_resid = robust_mad(resid)
            threshold = k * mad_resid
            mask = np.abs(resid) > threshold
            res.loc[mask.index] = mask.values
    except Exception as e:
        print(f"STL falló para {col}: {e}")
    return res

def rolling_jump_outliers(df, col='O3', window=ROLL_WINDOW_H, k=JUMP_K_MAD):
    """Detecta saltos bruscos de 1 hora comparando diff con MAD local."""
    res = pd.Series(False, index=df.index)
    s = df[col]
    diff = s.diff().abs()
    # MAD móvil de las diferencias
    roll_mad = diff.rolling(window=window, min_periods=1, center=True).apply(lambda x: robust_mad(x), raw=True)
    thr = k * roll_mad
    mask = (diff > thr) & (~diff.isna())
    res.loc[mask.index] = mask.values
    return res

def isolation_forest_outliers(df, features=None, contamination=ISO_CONTAMINATION):
    """Isolation Forest multivariado sobre las variables numéricas disponibles."""
    if features is None:
        candidates = ['O3', 'NO', 'NO2', 'NOx', 'Temp.', 'Veloc.', 'R.Sol.']
        features = [c for c in candidates if c in df.columns]
    res = pd.Series(False, index=df.index)
    sub = df[features].copy()
    valid_idx = sub.dropna(how='all').index
    if len(valid_idx) < 20:
        return res
    sub = sub.loc[valid_idx]
    try:
        # Relleno simple para poder correr IsolationForest
        sub = sub.ffill().bfill().fillna(0)
        iso = IsolationForest(contamination=contamination, random_state=RANDOM_STATE)
        pred = iso.fit_predict(sub.values)
        out_idx = sub.index[pred == -1]
        res.loc[out_idx] = True
    except Exception as e:
        print("IsolationForest falló:", e)
    return res

def neighbor_support(df, idx, col='O3', window=3, rel_tol=0.5):
    """
    Evalúa si un valor atípico tiene apoyo en sus vecinos temporales.
    Retorna True si el cambio respecto a la mediana vecina es pequeño en términos relativos.
    """
    half = (window - 1) // 2
    i_pos = df.index.get_loc(idx)
    start = max(0, i_pos - half)
    end = min(len(df) - 1, i_pos + half)
    vals = df[col].iloc[start:end+1].values
    center = df[col].iloc[i_pos]
    if np.isnan(center):
        return False
    if np.isnan(vals).sum() >= half + 1:
        return False
    med_neighbors = np.nanmedian(np.concatenate([vals[:half], vals[half+1:]])) if len(vals) > 1 else np.nan
    if np.isnan(med_neighbors):
        return False
    # Si la diferencia con la mediana vecina es menor que el valor absoluto * rel_tol, hay soporte
    if abs(center - med_neighbors) <= abs(center) * rel_tol:
        return True
    return False

def related_variables_spike(df, idx, variables, window=ROLL_WINDOW_H, k=2.0):
    """
    Comprueba si alguna de las variables relacionadas presenta también un pico
    (valor > k * MAD local) en el mismo instante.
    """
    i_pos = df.index.get_loc(idx)
    for var in variables:
        if var not in df.columns:
            continue
        val = df[var].iloc[i_pos]
        if pd.isna(val):
            continue
        # Mediana móvil centrada
        roll_med = df[var].rolling(window=window, min_periods=1, center=True).median()
        med = roll_med.iloc[i_pos]
        if pd.isna(med):
            continue
        mad = robust_mad(df[var].dropna())
        if mad == 0:
            continue
        if abs(val - med) > k * mad:
            return True
    return False

# -------------------------------------------------------------------
# PROCESAMIENTO DE UNA ESTACIÓN
# -------------------------------------------------------------------
def process_station(path, save_out=True, verbose=True):
    """
    Carga el CSV, aplica detección de outliers y clasifica en:
    - sensor_error: error físico o aislado sin apoyo vecinal.
    - evento_real: múltiples detectores y picos en variables relacionadas.
    - doubtful: casos ambiguos.
    Guarda un CSV con sufijo '_outliers' y devuelve el DataFrame procesado.
    """
    station_name = Path(path).stem
    if verbose:
        print(f"\nProcesando: {station_name}")

    # Cargar y estandarizar índice temporal
    df_raw = pd.read_csv(path)
    df = ensure_datetime_index(df_raw, datetime_col=None, freq='h')

    # Inicializar flags
    flags = pd.DataFrame(index=df.index)
    flags['physical_error'] = False
    flags['physical_reason'] = ""

    # 1. Comprobaciones físicas
    phys = physical_checks(df)
    flags['physical_error'] = phys['physical_error']
    flags['physical_reason'] = phys['physical_reason']

    # 2. Detección estadística (sobre O3 si existe)
    o3_col = 'O3' if 'O3' in df.columns else None
    if o3_col is None:
        print(f"  Aviso: {station_name} no tiene columna O3. Se saltan STL/jump/iso para O3.")
        stl_mask = pd.Series(False, index=df.index)
        jump_mask = pd.Series(False, index=df.index)
    else:
        stl_mask = stl_outliers(df, col=o3_col, period=PERIOD_H, k=STL_K_MAD)
        jump_mask = rolling_jump_outliers(df, col=o3_col, window=ROLL_WINDOW_H, k=JUMP_K_MAD)

    flags['outlier_stl'] = stl_mask
    flags['outlier_jump'] = jump_mask

    # 3. Isolation Forest multivariado (usa todas las variables numéricas)
    iso_mask = isolation_forest_outliers(df, features=None, contamination=ISO_CONTAMINATION)
    flags['outlier_iso'] = iso_mask

    # 4. Clasificación heurística: sensor_error / evento_real / doubtful
    flags['sensor_error'] = False
    flags['evento_real'] = False
    flags['doubtful'] = False

    related_vars = ['NO', 'NO2', 'NOx', 'Temp.', 'R.Sol.', 'Veloc.']

    for ts in df.index:
        # Si ya es error físico, se marca directamente como sensor_error
        if flags.at[ts, 'physical_error']:
            flags.at[ts, 'sensor_error'] = True
            continue

        is_stl = flags.at[ts, 'outlier_stl']
        is_jump = flags.at[ts, 'outlier_jump']
        is_iso = flags.at[ts, 'outlier_iso']
        any_det = is_stl or is_jump or is_iso
        if not any_det:
            continue

        n_det = sum([is_stl, is_jump, is_iso])

        # Soporte en vecinos (solo para O3)
        support = False
        if o3_col is not None and not pd.isna(df.at[ts, o3_col]):
            try:
                support = neighbor_support(df, ts, col=o3_col, window=3, rel_tol=0.6)
            except Exception:
                support = False

        # Picos en variables relacionadas
        spike_related = related_variables_spike(df, ts, related_vars, window=ROLL_WINDOW_H, k=2.0)

        # Reglas de decisión
        if n_det >= 2 and spike_related:
            flags.at[ts, 'evento_real'] = True
        else:
            if not support and n_det >= 1:
                flags.at[ts, 'sensor_error'] = True
            else:
                flags.at[ts, 'doubtful'] = True

    # 5. Preparar columna para imputación futura: O3_for_impute = O3 salvo donde sensor_error = True (se pone NaN)
    df_proc = df.copy()
    if o3_col is not None:
        df_proc['O3_for_impute'] = df_proc[o3_col].where(~flags['sensor_error'], np.nan)

    # Unir flags al DataFrame
    out_df = pd.concat([df_proc, flags], axis=1)

    # Guardar CSV
    if save_out:
        out_path = os.path.join(OUTPUT_DIR, f"{station_name}_outliers.csv")
        out_df.to_csv(out_path, index=True)
        if verbose:
            print(f"  Guardado: {out_path}")

    return out_df

# -------------------------------------------------------------------
# EJECUCIÓN SOBRE TODAS LAS ESTACIONES
# -------------------------------------------------------------------
all_doubtful = []
all_summaries = []

for path in station_paths:
    try:
        df_out = process_station(path, save_out=True, verbose=True)

        # Recopilar filas dudosas (doubtful)
        if 'doubtful' in df_out.columns:
            doubtful_rows = df_out[df_out['doubtful'] == True].copy()
            if not doubtful_rows.empty:
                doubtful_rows['station_file'] = Path(path).stem
                all_doubtful.append(doubtful_rows)

        # Resumen de conteos
        summary = {
            'station': Path(path).stem,
            'n_rows': len(df_out),
            'n_physical_error': int(df_out['physical_error'].sum()) if 'physical_error' in df_out.columns else 0,
            'n_outlier_stl': int(df_out['outlier_stl'].sum()) if 'outlier_stl' in df_out.columns else 0,
            'n_outlier_jump': int(df_out['outlier_jump'].sum()) if 'outlier_jump' in df_out.columns else 0,
            'n_outlier_iso': int(df_out['outlier_iso'].sum()) if 'outlier_iso' in df_out.columns else 0,
            'n_sensor_error': int(df_out['sensor_error'].sum()) if 'sensor_error' in df_out.columns else 0,
            'n_evento_real': int(df_out['evento_real'].sum()) if 'evento_real' in df_out.columns else 0,
            'n_doubtful': int(df_out['doubtful'].sum()) if 'doubtful' in df_out.columns else 0,
        }
        all_summaries.append(summary)

    except Exception as e:
        print(f"ERROR procesando {path}: {e}")

# Guardar casos dudosos combinados
if all_doubtful:
    combined_doubtful = pd.concat(all_doubtful, axis=0)
    doubtful_path = os.path.join(OUTPUT_DIR, "doubtful_cases_combined.csv")
    combined_doubtful.to_csv(doubtful_path, index=True)
    print(f"\nCasos dudosos combinados guardados en: {doubtful_path}")

# Guardar resumen global
if all_summaries:
    summary_df = pd.DataFrame(all_summaries)
    summary_path = os.path.join(OUTPUT_DIR, "outliers_summary.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"Resumen global guardado en: {summary_path}")

print("\nProceso completado. Revisa la carpeta:", OUTPUT_DIR)
Instrucciones de uso

Asegúrate de tener instaladas las librerías: pandas, numpy, scikit-learn, statsmodels.
Copia el código en un archivo .py o en una celda de Jupyter Notebook.
Ejecuta. Se crearán los archivos con sufijo _outliers.csv en la carpeta ~/Documents/GitHub/TFGFinal/Datos_TFG_outliers/.
Revisa el resumen outliers_summary.csv y, si existen, los casos dudosos en doubtful_cases_combined.csv.
Este código conserva todos los datos originales y añade columnas que indican la naturaleza de cada outlier. La columna O3_for_impute ya tiene NaN donde se ha considerado error de sensor, lista para el siguiente paso de imputación.
Ahora creame el segundo código
Ahora debemos crear el Código 2: Imputación de valores NaN según las especificaciones dadas. El usuario ha proporcionado detalles muy concretos:

Imputación por escala temporal de datos faltantes:
≤ 6h: interpolación temporal/lineal.
6h y <72h: imputación estacional por hora del día.
≥72h: imputación multivariante.
Quiere que se haga una imputación por estación para el método "por estaciones" y otra imputación para el método "todas las estaciones". Es decir, dos estrategias de imputación diferentes: una donde cada estación se imputa individualmente (usando solo sus propios datos) y otra donde se imputa utilizando todas las estaciones juntas (pooling). Luego, en los modelos, usaremos una u otra según el enfoque.
Además, nos pide reflexionar sobre cuál es mejor para cada caso. Pero en el código debemos implementar ambas opciones.
También tenemos que tener en cuenta que el código anterior (outliers) generó archivos con sufijo _outliers.csv que contienen la columna O3_for_impute donde los valores marcados como sensor_error ya son NaN. Además, en esos archivos se conservan todas las columnas originales más las flags. Para la imputación, debemos leer esos archivos, aplicar la imputación a todas las variables numéricas (no solo O3) según las reglas de duración del gap y generar dos conjuntos de datos: uno imputado por estación y otro imputado global (usando todas las estaciones). Luego guardaremos los datos imputados en nuevas carpetas.

También debemos considerar que hay variables categóricas (Estacion, Transecto) que no deben imputarse, y variables como Dist. y Angulo que son constantes por estación, pero pueden tener NaN? En el ejemplo, Dist y Angulo son constantes, pero si faltan, podríamos imputar con el valor constante de la estación. Pero según la descripción, la imputación debe ser para cada variable numérica. Angulo es numérico, pero es constante por estación, así que si falta, podría imputarse con el valor típico de esa estación (por ejemplo, la moda o el único valor). Pero como es constante, lo más sencillo es rellenar con el valor único (forward/backward fill) o con la media/moda de la estación. Para no complicar, podemos tratarlo como numérico y aplicar las mismas reglas, pero como es constante, cualquier método dará el mismo resultado.

Otra consideración: la imputación estacional por hora del día implica calcular la media (o mediana) de esa variable para cada hora del día a partir de los datos disponibles (sanos) de la propia estación (o de todas las estaciones en el caso global). Y luego rellenar los huecos largos con ese valor horario típico.

La imputación multivariante podría ser más compleja: podríamos usar un modelo de regresión (por ejemplo, Random Forest) para predecir la variable faltante a partir de otras variables. Pero el usuario dice "imputación multivariante", pero no especifica el método. Podríamos optar por una imputación por regresión lineal múltiple o usando un modelo iterativo como MICE. Sin embargo, para mantener la simplicidad y coherencia con el plazo, podríamos usar una imputación basada en la media condicional a otras variables (por ejemplo, usar KNN imputer o un modelo simple). Pero como es un código que debe ser ejecutable y no demasiado complejo, podemos implementar una imputación multivariante usando IterativeImputer de scikit-learn (que es experimental pero funciona) o KNNImputer. Otra opción es simplemente extender la imputación estacional incluyendo también el mes o algo así, pero no es multivariante.

El usuario pide reflexión, así que en los comentarios podemos explicar que para gaps >72h, lo ideal sería un modelo multivariante que aproveche la correlación con otras variables (NO, NO2, NOx, Temp, etc.), y proponemos usar IterativeImputer (MICE) con un estimador por defecto (BayesianRidge) o RandomForest. Esto puede ser computacionalmente costoso, pero para un dataset horario de varios años puede ser aceptable.

Además, debemos separar la imputación en dos versiones: por estación y global. En la versión global, agrupamos todas las estaciones y aplicamos la imputación como si fueran un solo conjunto, pero teniendo en cuenta que las variables como Estacion y Transecto son categóricas y podrían usarse como predictores en la imputación multivariante. En ese caso, deberíamos codificarlas (one-hot) antes de imputar. Pero para simplificar, podemos hacer la imputación global sin usar esas categóricas, o incluirlas codificadas. Dado que el código 2 viene después de outliers y antes de codificación, en este punto aún no hemos codificado las categóricas. Por lo tanto, para la imputación multivariante global, podríamos incluir la estación como variable numérica (por ejemplo, asignando un ID numérico a cada estación) o no incluirla. Para no complicar, podemos hacer la imputación multivariante global sin la estación, solo con las variables numéricas. Pero luego al juntar todas las estaciones, los patrones entre estaciones pueden ayudar a imputar. Otra opción es hacer la imputación multivariante por estación (cada estación por separado) pero usando también datos de otras estaciones? Eso sería más complejo.

Dado que el usuario quiere dos métodos de análisis (por estación y global), probablemente espera que en el método global se entrene un modelo con datos de todas las estaciones, y para ello los datos deben estar imputados de forma global (usando información de todas las estaciones). En cambio, en el método por estación, la imputación debe ser independiente para cada estación. Así que haremos:

Imputación por estación: para cada estación, aplicar las reglas de imputación usando solo los datos de esa estación.
Imputación global: concatenar todas las estaciones en un solo DataFrame, aplicar las mismas reglas de imputación pero ahora usando todos los datos disponibles (de todas las estaciones). Esto significa que para la imputación estacional por hora, se calculará la media por hora usando datos de todas las estaciones; para la multivariante, se usará un modelo entrenado con todas las estaciones.
Esto tiene sentido.

Ahora, para la imputación, necesitamos identificar gaps (secuencias consecutivas de NaN) en cada variable. Para cada gap, según su longitud, aplicamos un método. Esto implica iterar sobre cada variable y cada gap. Es importante no mezclar información futura: en la interpolación lineal usamos datos anteriores y posteriores, lo cual está bien porque es dentro de la misma serie y no implica usar datos futuros en el sentido de predicción (pero para imputación es aceptable). En la imputación estacional, usamos estadísticos de toda la serie (media por hora), lo cual también usa datos futuros, pero para imputación es común. En la imputación multivariante, si usamos un modelo entrenado con todos los datos, también usa datos futuros, pero es para rellenar. En general, la imputación puede hacerse con toda la serie porque no estamos modelando con partición temporal aún. La partición temporal se hará después. Así que es aceptable.

Implementación:
Leer cada archivo *_outliers.csv de la carpeta de salida del código 1.
Para cada archivo, identificar la estación y transecto (que están en las columnas). Pueden ser constantes en el archivo.
Definir las variables numéricas a imputar: NO, NO2, NOx, O3, Veloc., Direc., Temp., R.Sol., Dist., Angulo. (O3_for_impute será la que usemos para O3, ya que tiene los NaN de sensor_error; las demás variables aún pueden tener NaN originales). Pero en el código 1, solo creamos O3_for_impute, no modificamos las otras variables. Así que debemos imputar todas las variables numéricas, incluyendo O3 (usando O3_for_impute). Luego, al final, podemos renombrar O3_for_impute a O3.
Para la imputación por estación: procesar cada archivo individualmente.
Para la imputación global: concatenar todos los DataFrames (con una columna que identifique la estación), aplicar la imputación global y luego separar por estación para guardar archivos individuales? O guardar un único archivo global con todos los datos? El usuario quiere predicciones por estación, así que necesitamos mantener los datos separados por estación. Podemos guardar archivos individuales imputados globalmente (es decir, cada archivo contiene los datos de esa estación imputados con información global). Eso sería coherente: para el método global, cada estación tendrá sus datos imputados usando toda la información disponible de todas las estaciones.
Por tanto, generaremos dos carpetas de salida:

imputed_by_station/ con los archivos imputados individualmente.
imputed_global/ con los archivos imputados globalmente (uno por estación, pero la imputación se hizo con datos globales).
Procedimiento para imputar un DataFrame (df) con las reglas:

Para cada columna numérica en la lista:
Identificar los gaps (secuencias de NaN) usando máscara de NaN y diferenciando.
Para cada gap:
Longitud L.
Si L <= 6: usar interpolación lineal (con límite temporal, no hacia adelante, solo dentro del gap, usando pandas interpolate con method='linear' y limit_area='inside'?).
Si 6 < L < 72: imputación estacional por hora. Para ello, calcular la media (o mediana) de esa variable para cada hora del día (0-23) usando todos los valores no NaN de la serie (de la propia estación o de todas en global). Luego asignar a cada punto del gap el valor correspondiente a su hora.
Si L >= 72: imputación multivariante. Usaremos IterativeImputer de sklearn (necesita importar desde sklearn.experimental). Requiere que las variables tengan un orden y puede ser lento. Alternativa: usar KNNImputer que también es multivariante basado en similitud. KNNImputer es más simple y rápido. Podemos usar KNNImputer con n_neighbors=5. O podemos combinar: para cada gap grande, usar un imputador multivariante entrenado con datos completos de esa estación (o global) para predecir los valores faltantes. Pero KNNImputer no maneja bien gaps largos porque busca vecinos en el espacio de características, no en el tiempo. Para imputación temporal, sería mejor un modelo que use otras variables en el mismo instante, pero no la secuencia temporal. Eso es lo que hace KNNImputer: usa otras variables del mismo instante para imputar, asumiendo que hay correlación. Eso es válido si las variables están correlacionadas. También podríamos usar un modelo de regresión (RandomForest) entrenado con las otras variables. Pero para simplificar y porque es un código estándar, usaremos IterativeImputer con estimador BayesianRidge (por defecto) o RandomForestRegressor. IterativeImputer modela cada variable en función de las demás, de forma iterativa. Es más robusto pero más lento. Dado que tenemos muchas estaciones y variables, podría ser pesado. Podemos dejar configurable.
Otra opción: para gaps >=72, podríamos usar una imputación estacional extendida (por hora y mes) en lugar de multivariante. Pero el usuario pidió multivariante. Así que implementaremos una opción con KNNImputer o IterativeImputer. Para mantener la velocidad, usaré KNNImputer, que es más simple y no requiere configuración compleja. Además, KNNImputer puede usar tanto variables numéricas como (si codificamos) categóricas, pero aquí no las tenemos codificadas. Así que solo numéricas.

Pero KNNImputer no es específicamente temporal; sin embargo, como imputa basado en la similitud de las otras variables en el mismo instante, puede ser efectivo si hay correlación. También podríamos añadir variables temporales (hora, día) para ayudar, pero en este paso aún no las tenemos. Podríamos crearlas momentáneamente: extraer hora del datetime para la imputación estacional ya la usamos, y para la multivariante también podríamos incluir hora, día del año, etc., como predictores. Eso mejoraría la imputación. Pero para no complicar, lo dejamos como está.

Estrategia para imputación multivariante con KNNImputer:

Para cada gap largo, necesitamos imputar solo esos puntos. KNNImputer imputa todo el DataFrame a la vez, pero si lo aplicamos a todo, rellenará todos los NaN, incluyendo los pequeños, y no respetaría la jerarquía de métodos. Por lo tanto, debemos aplicar los métodos en orden: primero interpolación para cortos, luego estacional para medianos, y finalmente multivariante para largos, pero asegurándonos de que los NaN ya imputados en pasos anteriores no se usen como entrada para el multivariante (porque estarían completos). Lo más sencillo es procesar los gaps de mayor a menor complejidad? No, mejor aplicar secuencialmente: primero imputar cortos (≤6h) con interpolación, luego sobre los restantes NaN, identificar gaps medianos (6-72h) y aplicar estacional, luego sobre los restantes, aplicar multivariante para los largos (≥72h). Pero tras la interpolación, los gaps cortos desaparecen, pero pueden quedar gaps largos que ahora son más pequeños? No, porque la interpolación solo rellena los cortos, los largos siguen siendo largos. Después de la estacional, los medianos se rellenan, y los largos pueden haberse convertido en medianos si se rellenaron partes? No, porque la estacional rellena todo el gap mediano, no parcialmente. Así que podemos hacerlo secuencialmente.
Para la imputación estacional, necesitamos la media por hora. Para cada gap, asignamos a cada fila el valor medio de esa hora calculado a partir de los datos no NaN de la serie (excluyendo el propio gap). Debemos calcularlo antes de imputar. Es importante que el cálculo de la media por hora no incluya los valores que vamos a imputar (para evitar sesgo). Podemos calcularlo sobre toda la serie original (con NaN) pero excluyendo los puntos del gap actual. Pero como hay múltiples gaps, lo más fácil es calcular la media por hora usando todos los valores no NaN de la serie (incluyendo otros gaps que aún no se han imputado, pero esos son NaN, así que no afectan). Eso es correcto.

Para la imputación multivariante con KNNImputer, necesitamos un conjunto de variables predictoras. Usaremos todas las demás variables numéricas disponibles (incluyendo posiblemente las temporales). Pero en este punto, las variables temporales no están creadas. Podríamos crear columnas de hora y día del año para mejorar la imputación, pero no es obligatorio. Para mantener la independencia del código, podemos simplemente usar las variables numéricas originales. KNNImputer necesita que no haya NaN en las predictoras, pero las variables que estamos imputando tienen NaN. KNNImputer internamente maneja NaN usando distancias basadas en valores observados. Es robusto.

Pero KNNImputer imputa todo el conjunto a la vez. Si lo aplicamos solo a las filas con gaps largos, podemos seleccionar un subconjunto. Sin embargo, KNNImputer usa vecinos que pueden ser de cualquier fila. Si aplicamos a todo el DataFrame, imputará todos los NaN, incluyendo los que ya habíamos imputado con otros métodos (y que ahora son valores). Para evitar eso, podemos hacer una copia, aplicar los primeros dos métodos, y luego sobre las filas que aún tienen NaN (los largos) aplicar KNNImputer pero entrenando con todas las filas que tienen valores completos en las predictoras (incluyendo las filas imputadas anteriormente). Eso es razonable porque esas filas imputadas ya tienen valores estimados que pueden servir como referencia. Pero cuidado: si usamos filas imputadas con métodos simples para entrenar el modelo multivariante, podríamos propagar errores. Una alternativa más limpia es entrenar el imputador multivariante solo con filas originalmente completas (o con imputaciones previas solo de otros métodos, pero sería enredo). Para simplificar, podemos aplicar KNNImputer a todo el DataFrame después de los dos primeros pasos, pero eso imputaría también los NaN que quedan, que son los largos. Pero también podría modificar ligeramente algunos valores ya imputados si no se configuran como máscara. Sin embargo, KNNImputer no modifica valores no NaN. Así que podemos aplicarlo a todo el DataFrame después de los dos primeros pasos, y solo afectará a los NaN restantes. Eso es seguro y sencillo.

Procedimiento general para un DataFrame (df) con índice datetime:
Crear una copia df_imputed = df.copy().
Para cada variable numérica en columnas_num:
Obtener serie original (con NaN).
Identificar gaps de NaN usando:
mask = serie.isna()
Identificar grupos consecutivos

groups = mask.ne(mask.shift()).cumsum()
gap_groups = groups[mask].value_counts()
Para cada gap (grupo) con longitud L:
indices = grupo.index
if L <= 6:
Interpolación lineal

Usar pandas interpolate en la serie, pero solo para esos índices? Podemos hacer:

serie_interp = serie.interpolate(method='linear', limit_area='inside', limit_direction='both')
df_imputed.loc[indices, var] = serie_interp.loc[indices]
elif L < 72:
Imputación estacional por hora

Calcular media por hora usando todos los datos no NaN (de la serie original)

hour = indices.hour # asumiendo índice datetime
Calcular medias por hora (excluyendo los valores del gap actual? Pero es complicado, usamos toda la serie original)

Para evitar usar el propio gap, podemos calcular medias excluyendo los índices del gap.

Sin embargo, como el gap son NaN, no influyen. Pero si hay otros gaps, esos son NaN, tampoco influyen.

Así que podemos usar toda la serie original no NaN.

hourly_mean = serie.dropna().groupby(serie.dropna().index.hour).mean()
Asignar

df_imputed.loc[indices, var] = indices.hour.map(hourly_mean).values
else:
Marcar para imputación multivariante posterior

No hacemos nada aquí, los NaN permanecen

pass
Después de procesar todas las variables con los dos primeros métodos, aplicamos KNNImputer a todo el DataFrame (solo columnas numéricas) para imputar los NaN restantes (que son los gaps >=72h).
Seleccionar columnas numéricas.
Crear imputer = KNNImputer(n_neighbors=5)
Ajustar y transformar: df_imputed[num_cols] = imputer.fit_transform(df_imputed[num_cols])
Esto imputará todos los NaN restantes.
Nota: Para la imputación estacional, hemos usado la media por hora. Podríamos usar mediana para ser más robusto. Dejaremos configurable.

Además, debemos considerar que algunas variables como Direc. y Angulo son circulares, la media lineal no es apropiada. Para ángulos, deberíamos usar la media circular. Pero como son constantes por estación (probablemente), la media dará el mismo valor. Si hay variabilidad, la media lineal puede ser engañosa. Para simplificar, asumimos que son constantes o que la variación es pequeña. De todas formas, en la imputación multivariante, KNNImputer usará distancia euclidiana, lo cual no es adecuado para ángulos. Pero dada la naturaleza de los datos (ángulos de viento y dirección de transecto), probablemente sean constantes o tengan poca variación. Si no, habría que transformarlos a seno/coseno antes de imputar. Pero el código 2 es antes de la codificación. Podríamos, en este paso, transformar temporalmente las variables circulares a seno/coseno para la imputación multivariante, y luego revertir. Eso complica. Lo dejamos así, asumiendo que no es un gran problema.

Para la imputación global, el procedimiento es similar pero trabajando con un DataFrame concatenado que incluye todas las estaciones. Para identificar gaps, debemos considerar que los gaps pueden estar separados por cambios de estación. Si concatenamos sin más, se crearán gaps artificiales en los bordes entre estaciones. Debemos asegurarnos de que la imputación no mezcle datos de diferentes estaciones en la misma secuencia temporal. Para evitar eso, podemos agrupar por estación dentro del DataFrame global y aplicar la imputación a cada grupo por separado, pero usando estadísticos globales (por ejemplo, la media por hora calculada globalmente). Sin embargo, la imputación multivariante global podría beneficiarse de usar datos de otras estaciones para predecir valores en una estación. Para ello, necesitamos que el imputador vea todas las filas juntas. Pero al concatenar, las filas de diferentes estaciones se mezclan en el tiempo, lo cual no es correcto porque no hay continuidad temporal entre estaciones. La imputación multivariante basada en similitud de otras variables sí puede usar filas de otras estaciones en el mismo instante (si tienen la misma hora) porque las condiciones meteorológicas pueden estar correlacionadas espacialmente. Eso es válido. Por lo tanto, podemos concatenar directamente, pero debemos tener cuidado de no crear falsos gaps en los límites entre estaciones debido a que el índice datetime no es continuo (cada estación tiene su propio rango). Si concatenamos, el índice será el datetime, pero habrá múltiples filas con el mismo datetime (una por estación). Al hacer un stack, tendremos un DataFrame con índice datetime y múltiples filas por hora. Esto es correcto. Para identificar gaps, no debemos considerar la secuencia temporal porque ahora cada estación es independiente; la continuidad temporal es por estación. Así que no podemos tratar todo el DataFrame como una serie temporal única. Por lo tanto, debemos aplicar la imputación por estación dentro del DataFrame global, pero con estadísticos globales y con imputación multivariante que puede usar filas de otras estaciones.

Enfoque para imputación global:

Concatenar todos los DataFrames de las estaciones en uno solo, añadiendo una columna 'station_id' para identificar.
Luego, para la imputación, necesitamos manejar cada variable por separado, pero teniendo en cuenta que los NaN pueden estar en cualquier estación.
Para la interpolación lineal (≤6h): no podemos aplicar directamente porque la serie no es continua entre estaciones. Debemos agrupar por estación y aplicar interpolación dentro de cada grupo. Así que la interpolación se hará por estación.
Para la imputación estacional (6-72h): la media por hora se calculará globalmente (usando datos de todas las estaciones) pero se asignará a cada gap dentro de su estación. Así que para cada gap en una estación, usamos la media global de esa hora.
Para la imputación multivariante (≥72h): queremos usar un imputador que aprenda de todas las estaciones. Podemos aplicar KNNImputer a todo el DataFrame (sin agrupar), pero incluyendo la estación como variable numérica (codificada como entero) para que el imputador pueda diferenciar. O podríamos no incluirla y confiar en que las relaciones entre variables son similares entre estaciones. Para mejorar, podemos incluir el ID de estación como predictor. Pero como es categórica, habría que codificarla. Podríamos hacer one-hot, pero eso aumenta dimensiones. Una opción simple es asignar un número a cada estación y usarlo como variable numérica. KNNImputer tratará ese número como una característica más, lo que puede ayudar a agrupar por estación. No es ideal, pero funciona.
Otra opción: hacer la imputación multivariante global pero agrupada por estación, pero permitiendo que el modelo se entrene con datos de todas las estaciones y luego se aplique a cada estación por separado. Eso sería más complicado. Para simplificar, asumiremos que las relaciones son homogéneas y aplicaremos KNNImputer a todo el conjunto, incluyendo la estación como variable numérica.

Implementaremos:

Para interpolación y estacional: loop sobre estaciones (groupby) y aplicar dentro de cada grupo.
Para multivariante: después de los dos primeros pasos, tendremos un DataFrame con algunos NaN restantes. Aplicamos KNNImputer a todas las columnas numéricas (incluyendo la variable 'station_id' numérica). Pero station_id no debería tener NaN. Luego, separamos por estación y guardamos.
Nota: En la imputación estacional global, la media por hora se calcula sobre todas las estaciones. Esto significa que si una estación tiene un patrón horario diferente, se verá influenciada por las demás. Puede ser deseable o no. Pero como es "global", está bien.

También debemos decidir si la variable 'O3' a imputar es la columna 'O3' original o 'O3_for_impute'. En los archivos de outliers, tenemos ambas. Para la imputación, usaremos 'O3_for_impute' como la variable O3 a imputar, y luego al final podemos renombrarla a 'O3' y eliminar la original. O simplemente trabajar con 'O3_for_impute' y luego en los modelos usar esa columna. Pero para mantener consistencia, podemos renombrar.

Vamos a diseñar el código 2 con funciones modulares.

Estructura del código:

Importaciones.
Parámetros configurables (rutas, umbrales, métodos).
Definir columnas numéricas.
Función para imputar un DataFrame (de una estación) con los métodos secuenciales, pero parametrizada para usar estadísticos locales o globales (medias por hora). Para la versión por estación, las medias se calculan dentro de la función con los datos de esa estación. Para la versión global, necesitamos pasar las medias por hora globales y luego aplicar la imputación dentro de cada estación.
Función para imputar todas las estaciones en modo "by_station": leer cada archivo, imputar con estadísticos locales, guardar en carpeta imputed_by_station.
Función para imputar en modo "global": concatenar todos los DataFrames, añadir station_id, calcular medias por hora globales, luego iterar sobre estaciones y para cada una aplicar imputación (interpolación y estacional) usando las medias globales. Luego, al final, aplicar KNNImputer global a todo el conjunto concatenado (con station_id) para los largos. Finalmente, separar por estación y guardar en imputed_global.
La imputación multivariante global se aplica al DataFrame concatenado después de haber hecho la interpolación y estacional en cada estación. Eso significa que los NaN restantes son los largos de todas las estaciones. KNNImputer se aplica a todas las filas juntas, usando todas las variables numéricas incluyendo station_id. Luego, las filas imputadas se reasignan a sus estaciones.

Debemos tener cuidado de que al aplicar KNNImputer, no debe modificar valores ya imputados, pero como solo actúa sobre NaN, está bien.

Implementaremos con KNNImputer de sklearn.impute.

Ahora, redactemos el código con explicaciones.

Además, añadiremos un paso de verificación de que los archivos de outliers existen. Usaremos la misma lista de nombres de estaciones pero construyendo las rutas de salida del código 1. Asumimos que los archivos están en OUTPUT_DIR del código 1 (la carpeta outliers). El usuario puede haber ejecutado el código 1 y tenerlos allí. Usaremos esa carpeta como entrada.

El código generará dos nuevas carpetas dentro de la misma ruta base (por ejemplo, junto a outliers). Pondremos:

imputed_by_station
imputed_global
Dentro de cada una, guardaremos los CSV con el mismo nombre de archivo original (sin sufijo _outliers, o quizás con sufijo _imputed_by_station y _imputed_global). Para mantener consistencia, podemos usar el nombre base de la estación (ej. T1_E1_Alicante.csv) pero en las subcarpetas. Así, el archivo original se puede identificar.

También guardaremos un resumen de la imputación (opcional).

Ahora, escribamos el código.
python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 2: Imputación de valores NaN según duración del gap
y dos estrategias: por estación individual y global (todas las estaciones).

Requisitos:
- pandas, numpy, scikit-learn (para KNNImputer)
- Los archivos de entrada son los generados en el Código 1 (*_outliers.csv)
  que contienen la columna 'O3_for_impute' con los NaN de errores de sensor.
"""

import os
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.impute import KNNImputer

# ============================================================================
# PARÁMETROS CONFIGURABLES
# ============================================================================

# Carpeta donde están los archivos con outliers (salida del Código 1)
INPUT_DIR = os.path.expanduser("~/Documents/GitHub/TFGFinal/Datos_TFG_outliers")

# Carpetas de salida para los datos imputados
OUTPUT_BY_STATION = os.path.join(INPUT_DIR, "imputed_by_station")
OUTPUT_GLOBAL = os.path.join(INPUT_DIR, "imputed_global")
os.makedirs(OUTPUT_BY_STATION, exist_ok=True)
os.makedirs(OUTPUT_GLOBAL, exist_ok=True)

# Umbrales para los métodos de imputación (en horas)
SHORT_GAP_MAX = 6          # ≤ 6h → interpolación lineal
MEDIUM_GAP_MAX = 72        # >6h y <72h → estacional por hora
LONG_GAP_MIN = 72          # ≥72h → multivariante (KNN)

# Método para la imputación estacional: 'mean' o 'median'
SEASONAL_STAT = 'mean'

# Columnas numéricas a imputar (incluye O3_for_impute, que será la O3 final)
NUM_COLS = [
    'NO', 'NO2', 'NOx', 'O3_for_impute',   # O3 se imputa desde esta columna
    'Veloc.', 'Direc.', 'Temp.', 'R.Sol.',
    'Dist.', 'Angulo'
]

# Columnas que se conservan pero no se imputan (categóricas, flags, etc.)
# (Se mantienen intactas en el output)

# Para KNNImputer (multivariante)
KNN_NEIGHBORS = 5

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def load_station_data(filename):
    """Carga un archivo CSV de una estación (con índice datetime)."""
    df = pd.read_csv(filename, index_col=0, parse_dates=True)
    # Asegurar que el índice es datetime
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)
    return df

def get_gap_lengths(series):
    """
    Devuelve un diccionario con los grupos de NaN y su longitud.
    Cada grupo es un objeto Index con las posiciones del gap.
    """
    mask = series.isna()
    # Identificar grupos consecutivos de True (NaN)
    groups = mask.ne(mask.shift()).cumsum()
    gap_groups = {}
    for g in groups[mask].unique():
        idx = groups[groups == g].index
        gap_groups[tuple(idx)] = len(idx)   # guardamos el índice como tupla para poder iterar
    return gap_groups

def impute_by_station(df, use_global_hourly_stats=None):
    """
    Imputa un DataFrame de una sola estación aplicando los métodos en orden:
    1. Gaps cortos (≤6h) → interpolación lineal.
    2. Gaps medios (6-72h) → media/mediana por hora (estacional).
    3. Gaps largos (≥72h) → se dejan para KNN (se aplica después externamente).

    Si use_global_hourly_stats es un dict (hour -> valor), se usan esas medias globales;
    de lo contrario, se calculan a partir de los propios datos de la estación.
    """
    df_imp = df.copy()
    # Procesar cada columna numérica
    for col in NUM_COLS:
        if col not in df_imp.columns:
            continue
        serie = df_imp[col]
        if serie.isna().sum() == 0:
            continue

        # Obtener gaps
        gaps = get_gap_lengths(serie)
        for idx_tuple, L in gaps.items():
            idx = pd.Index(idx_tuple)  # convertir a Index
            if L <= SHORT_GAP_MAX:
                # Interpolación lineal
                # interpolate con limit_direction='both' rellena usando valores anteriores y posteriores
                serie_interp = serie.interpolate(method='linear', limit_direction='both')
                df_imp.loc[idx, col] = serie_interp.loc[idx]
            elif L < MEDIUM_GAP_MAX:
                # Imputación estacional por hora
                if use_global_hourly_stats is not None:
                    # Usar estadísticos globales
                    hourly_vals = idx.hour.map(use_global_hourly_stats)
                else:
                    # Calcular estadísticos por hora de la propia estación
                    # Excluimos los valores del gap actual (aunque son NaN, no influyen)
                    if SEASONAL_STAT == 'mean':
                        hourly_stats = serie.dropna().groupby(serie.dropna().index.hour).mean()
                    else:
                        hourly_stats = serie.dropna().groupby(serie.dropna().index.hour).median()
                    hourly_vals = idx.hour.map(hourly_stats)
                df_imp.loc[idx, col] = hourly_vals.values
            else:
                # Gaps largos: se dejarán para KNN (no hacemos nada aquí)
                pass
    return df_imp

def apply_knn_imputation(df, num_cols, extra_cols=None):
    """
    Aplica KNNImputer a las columnas numéricas del DataFrame.
    extra_cols: columnas adicionales que se incluirán como predictores pero no se imputarán (ej. station_id).
    """
    cols_to_impute = [c for c in num_cols if c in df.columns]
    if extra_cols:
        all_cols = cols_to_impute + extra_cols
    else:
        all_cols = cols_to_impute

    # Seleccionar subconjunto
    sub = df[all_cols].copy()
    # KNNImputer solo funciona con datos numéricos; ya lo son.
    imputer = KNNImputer(n_neighbors=KNN_NEIGHBORS)
    imputed_array = imputer.fit_transform(sub)
    # Reasignar solo las columnas imputadas (las extra no se modifican)
    df_imp = df.copy()
    df_imp[cols_to_impute] = imputed_array[:, :len(cols_to_impute)]
    return df_imp

# ============================================================================
# PROCESAMIENTO POR ESTACIÓN (INDIVIDUAL)
# ============================================================================

def process_by_station():
    print("\n=== Imputación por estación (individual) ===")
    # Buscar todos los archivos _outliers.csv en INPUT_DIR
    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        print(f"Procesando {station_name} ...")
        df = load_station_data(filepath)
        # Imputar con estadísticos propios
        df_imp = impute_by_station(df, use_global_hourly_stats=None)
        # Aplicar KNN a los NaN restantes (gaps largos)
        # Nota: KNN se aplica a todas las filas, pero solo modificará NaN
        df_imp = apply_knn_imputation(df_imp, NUM_COLS)
        # Renombrar O3_for_impute a O3 para los modelos
        if 'O3_for_impute' in df_imp.columns:
            df_imp.rename(columns={'O3_for_impute': 'O3'}, inplace=True)
        # Guardar
        out_path = os.path.join(OUTPUT_BY_STATION, f"{station_name}.csv")
        df_imp.to_csv(out_path, index=True)
        print(f"  Guardado en {out_path}")

# ============================================================================
# PROCESAMIENTO GLOBAL (TODAS LAS ESTACIONES JUNTAS)
# ============================================================================

def process_global():
    print("\n=== Imputación global (todas las estaciones) ===")
    outlier_files = list(Path(INPUT_DIR).glob("*_outliers.csv"))
    if not outlier_files:
        print(f"No se encontraron archivos *_outliers.csv en {INPUT_DIR}")
        return

    # Lista para concatenar
    dfs = []
    station_names = []
    for filepath in outlier_files:
        station_name = filepath.stem.replace("_outliers", "")
        station_names.append(station_name)
        df = load_station_data(filepath)
        df['station_id'] = station_name  # identificador de estación
        dfs.append(df)

    # Concatenar todos
    df_all = pd.concat(dfs, axis=0)
    # Ordenar por datetime y station_id? No es necesario, pero para KNN puede ayudar
    # Dejamos como está.

    # 1. Calcular estadísticos horarios globales (media/mediana por hora) para cada variable
    #    usando todos los datos no NaN de todas las estaciones.
    global_hourly_stats = {}
    for col in NUM_COLS:
        if col not in df_all.columns:
            continue
        s = df_all[col].dropna()
        if SEASONAL_STAT == 'mean':
            hourly = s.groupby(s.index.hour).mean()
        else:
            hourly = s.groupby(s.index.hour).median()
        global_hourly_stats[col] = hourly

    # 2. Imputar cada estación por separado usando los estadísticos globales
    #    (para interpolación y estacional). Hacemos loop sobre las estaciones.
    df_all_imp = df_all.copy()
    for station in station_names:
        mask = df_all_imp['station_id'] == station
        df_station = df_all_imp.loc[mask].copy()
        # Aplicar imputación con estadísticos globales (pasamos el dict de medias por hora)
        # Necesitamos una versión de impute_by_station que acepte un dict con las medias por columna.
        # Modificamos la función para que acepte un dict de series indexadas por hora.
        # Para simplificar, reimplementamos aquí un bucle específico.
        for col in NUM_COLS:
            if col not in df_station.columns:
                continue
            serie = df_station[col]
            if serie.isna().sum() == 0:
                continue
            gaps = get_gap_lengths(serie)
            for idx_tuple, L in gaps.items():
                idx = pd.Index(idx_tuple)
                if L <= SHORT_GAP_MAX:
                    # Interpolación lineal dentro de la estación (usamos la serie completa de la estación)
                    serie_interp = serie.interpolate(method='linear', limit_direction='both')
                    df_station.loc[idx, col] = serie_interp.loc[idx]
                elif L < MEDIUM_GAP_MAX:
                    # Usar estadístico global por hora
                    hourly_vals = idx.hour.map(global_hourly_stats[col])
                    df_station.loc[idx, col] = hourly_vals.values
                # largos se dejan para KNN
        # Actualizar el DataFrame global
        df_all_imp.loc[mask] = df_station

    # 3. Aplicar KNN imputer a todo el conjunto (incluyendo station_id como predictor)
    #    para los gaps largos restantes.
    #    Incluimos station_id como variable numérica (convertimos a código)
    #    Primero, convertir station_id a código numérico (0,1,2,...)
    df_all_imp['station_code'] = pd.factorize(df_all_imp['station_id'])[0]
    extra_cols = ['station_code']
    # Aplicar KNN
    df_all_imp = apply_knn_imputation(df_all_imp, NUM_COLS, extra_cols=extra_cols)
    # Eliminar station_code (ya no necesario)
    df_all_imp.drop(columns=['station_code'], inplace=True)

    # 4. Separar por estación y guardar
    for station in station_names:
        df_station = df_all_imp[df_all_imp['station_id'] == station].copy()
        df_station.drop(columns=['station_id'], inplace=True)
        # Renombrar O3_for_impute a O3
        if 'O3_for_impute' in df_station.columns:
            df_station.rename(columns={'O3_for_impute': 'O3'}, inplace=True)
        out_path = os.path.join(OUTPUT_GLOBAL, f"{station}.csv")
        df_station.to_csv(out_path, index=True)
        print(f"  Guardado {station} en {out_path}")

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("Iniciando imputación de datos...")
    process_by_station()
    process_global()
    print("\nProceso completado. Revise las carpetas:")
    print(f"  - {OUTPUT_BY_STATION}")
    print(f"  - {OUTPUT_GLOBAL}")
Instrucciones de uso

Asegúrate de que el Código 1 se haya ejecutado correctamente y que los archivos *_outliers.csv estén en la carpeta Datos_TFG_outliers.
Instala las dependencias si es necesario: pip install pandas numpy scikit-learn.
Ejecuta este script. Creará dos subcarpetas:

imputed_by_station: datos imputados usando solo información de cada estación.
imputed_global: datos imputados usando información combinada de todas las estaciones (estadísticos horarios globales y KNN multivariante global).
En ambos casos, la columna objetivo se renombra a O3 (lista para los modelos) y se conservan todas las demás columnas originales (flags, variables categóricas, etc.).
Notas importantes

La imputación respeta los umbrales definidos (6h, 72h) y los métodos solicitados.
Para gaps ≥72h se utiliza KNNImputer con 5 vecinos, incluyendo la estación como variable numérica (solo en el caso global) para ayudar a la imputación.
Las variables circulares (Direc., Angulo) se tratan como numéricas lineales; si se desea un tratamiento más preciso, se podría modificar en futuros pasos.
Si alguna columna de NUM_COLS no existe en un archivo, se omite sin errores.
Ahora creame el siguiente código COMPLETO listo para copiar, pegar y ejecutar
Vamos a crear el Código 3, que se encarga de la codificación de variables temporales (datetime) en seno/coseno, la transformación de las variables direccionales (Direc., Angulo) también a seno/coseno, y la codificación de las variables categóricas (Estacion, Transecto) mediante OneHotEncoder (para modelos de árbol) o Embedding (para redes neuronales). Dado que los embeddings se aprenden durante el entrenamiento, en este paso solo prepararemos las variables para su uso posterior; es decir, para los modelos de deep learning dejaremos las categorías como enteros (índices) para que luego se puedan usar con capas Embedding. Para los modelos de árbol aplicaremos OneHotEncoder directamente y guardaremos las columnas resultantes.

Además, este código debe trabajar sobre los datasets imputados generados en el paso anterior (tanto por estación como global). Como hay dos estrategias de imputación (individual y global), también deberemos generar dos conjuntos de datos codificados (pero con la misma estructura de codificación). También debemos considerar que la variable "Estacion" en el dataset global ya está presente como columna, y en los datasets individuales también está presente pero es constante (podría eliminarse o dejarse). Seguiremos el mismo esquema de salida: una carpeta para los datos codificados.

Asegurémonos de que el código sea claro y completo.

Estructura del código:
Definir parámetros de entrada/salida.
Función para codificación temporal: extraer hora, día, semana, mes, año y convertirlos a seno/coseno.
Función para codificación direccional: convertir Direc. y Angulo a seno/coseno (recordar que están en grados, de 0 a 360). Se generan dos columnas por cada variable: var_sin, var_cos.
Función para codificación de categóricas: para ML (árboles) usar OneHotEncoder; para DL (redes) dejar como enteros (label encoding) y guardar un mapeo para luego usarlo en embeddings.
Aplicar a todos los archivos en las carpetas de entrada (imputed_by_station y imputed_global). Para cada archivo, generar dos versiones: una para ML (con one-hot) y otra para DL (con enteros). Pero tal vez sea más simple generar un solo archivo con las categóricas en formato entero y luego en los códigos de modelos hacer el one-hot específico. Sin embargo, para evitar confusiones, generaremos dos carpetas separadas: "encoded_ml" y "encoded_dl". Dentro de cada una, replicaremos la estructura de estaciones (individual/global). Es decir, tendremos:
encoded_ml/by_station/
encoded_ml/global/
encoded_dl/by_station/
encoded_dl/global/
Dentro de cada subcarpeta, los archivos CSV con las columnas ya transformadas.

Para la codificación de categóricas en ML, haremos one-hot de las columnas "Estacion" y "Transecto". En los datasets individuales, "Estacion" es constante (solo un valor), entonces el one-hot producirá una sola columna (por ejemplo, Estacion_Estacion3). Eso es aceptable. En el dataset global, habrá múltiples estaciones y transectos.

Para la codificación en DL, dejaremos "Estacion" y "Transecto" como enteros (0,1,2,...). También necesitamos guardar los diccionarios de mapeo (por ejemplo, para saber qué índice corresponde a cada estación), porque luego en la predicción puede ser útil. Los guardaremos en archivos JSON dentro de la carpeta correspondiente.

Además, las variables temporales y direccionales se transforman igual para ambos enfoques.

El código también debe preservar las columnas numéricas originales (NO, NO2, NOx, O3, Veloc., Temp., R.Sol., Dist.) y las columnas de flags o metadatos? Probablemente las flags ya no son necesarias para los modelos. Podemos eliminarlas para no ensuciar. Pero las dejaremos por si acaso, o mejor las eliminamos. En los códigos de modelos, las variables predictoras serán todas las numéricas + las nuevas cíclicas + las categóricas codificadas. Las flags (sensor_error, evento_real, doubtful, outlier_*, etc.) no deben usarse como entrada. Así que en este paso podemos filtrar las columnas que no sean predictoras. Definamos una lista de columnas predictoras:

Numéricas originales: NO, NO2, NOx, O3, Veloc., Direc., Temp., R.Sol., Dist., Angulo (pero O3 es el target, debe mantenerse para poder separarlo después; en los modelos se usa como y, pero en el dataset está).
Nuevas cíclicas: hour_sin, hour_cos, day_sin, day_cos, week_sin, week_cos, month_sin, month_cos, year (año no es cíclico, pero lo podemos dejar como entero; sin embargo el año puede ser útil para tendencias. Lo dejamos como numérico lineal. También podríamos codificarlo seno/coseno si hubiera periodicidad anual, pero no es estrictamente necesario. Lo dejamos como escalar).
Direccionales: Direc._sin, Direc._cos, Angulo_sin, Angulo_cos.
Categóricas codificadas.
Pero debemos tener en cuenta que las originales Direc. y Angulo ya no son necesarias una vez creadas sus versiones seno/coseno. Podemos eliminarlas para evitar multicolinealidad. Así que las reemplazamos.

Además, la variable O3 es el target, así que la conservamos.

También existen las columnas "Estacion" y "Transecto" originales (texto) que se eliminarán después de la codificación.

Entonces, el flujo para cada archivo:

Leer CSV.
Asegurar índice datetime.
Extraer componentes de datetime: hour, day, week, month, year.
Aplicar transformación seno/coseno a hour, day, week, month. Nota: day se refiere al día del mes (1-31) o día del año? Especificaste "día", pero en el contexto de periodicidad mensual, el día del mes tiene ciclo mensual, no anual. Para capturar la estacionalidad dentro del mes, podría ser día del mes (1-31). Pero la mayoría de modelos usan día del año (1-366) para estacionalidad anual. Como dijiste "día" sin especificar, podemos incluir ambas: día del mes y día del año. Pero para no complicar, haremos día del año (dayofyear) para capturar la estacionalidad anual, y también día de la semana (weekday) que ya está en la semana. En tu lista incluiste "semana", así que usaremos semana del año (weekofyear). Entonces:
hour: 0-23 -> seno/coseno con período 24.
dayofyear: 1-366 (o 1-365) -> período 365.25. Usamos 365.
week: 1-52 (o 53) -> período 52.
month: 1-12 -> período 12.
year: se deja como numérico (no cíclico).
Para las variables direccionales: dirección en grados (0-360). Convertir a radianes y luego sin, cos.
Para categóricas:
Para ML: aplicar OneHotEncoder (de sklearn) y añadir columnos dummies. Eliminar las columnas originales.
Para DL: usar LabelEncoder (o simplemente factorize) para convertir a enteros. Guardar diccionario con el mapeo.
Eliminar columnas originales Direc., Angulo, Estacion, Transecto (las reemplazadas).
Guardar archivo en la carpeta correspondiente.
También debemos considerar que en el dataset global, la columna "Estacion" tiene valores que pueden no estar presentes en todos los archivos; pero al aplicar one-hot se generarán columnas para todas las estaciones presentes en ese archivo. Para que sea consistente, podríamos ajustar el one-hot a nivel global (usando todas las estaciones posibles). Pero como estamos procesando archivo por archivo, es más sencillo dejarlo así. Sin embargo, para el dataset global, al concatenar todas las estaciones, la columna "Estacion" tendrá varios valores; al hacer one-hot sobre ese archivo global se generarán columnas para todas las estaciones presentes. Eso es correcto. Para los datasets individuales, solo aparecerá una estación, así que se creará una columna dummy para esa estación.

Para el caso DL, el mapeo debe ser consistente entre archivos si queremos usar embeddings. Podemos crear un mapeo global de estaciones y transectos a partir de todos los archivos. Pero como los datasets se procesan por separado, podemos generar el mapeo a partir de los datos globales y luego aplicarlo a los individuales. Para simplificar, en este código haremos que para DL, tanto en individual como global, se use el mismo mapeo (calculado sobre todas las estaciones disponibles). Para ello, primero leeremos todos los archivos (de una estrategia, por ejemplo la global) para obtener las categorías únicas. Pero como hay dos estrategias de imputación (individual y global) y queremos mantener separadas, tal vez sea más sencillo que los datasets individuales mantengan su propio mapeo (solo ellos) y los globales otro. No habrá problema porque luego en el entrenamiento, los modelos por estación usarán solo los datos de esa estación, y los globales usarán todas.

Para mantenerlo simple, haremos que en la carpeta encoded_dl, por cada archivo se guarden los enteros con un mapeo propio (factorize). Adicionalmente, guardaremos un archivo JSON con el mapeo para ese archivo. Pero para los modelos globales, necesitaremos un mapeo único para todas las estaciones; al procesar el archivo global concatenado, factorize generará índices consistentes dentro de ese archivo. Eso es suficiente.

Por tanto, en el código, para la versión DL simplemente aplicaremos pd.factorize a las columnas categóricas y guardaremos también el mapeo en un JSON aparte (opcional). Lo implementaremos.

Ahora, escribamos el código completo.

python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 3: Codificación de variables temporales (datetime) en seno/coseno,
          transformación de dirección y ángulo a seno/coseno,
          y codificación de variables categóricas (Estacion, Transecto)
          para ML (OneHot) y DL (enteros para embedding).

Entrada: Carpetas con datos imputados:
    - Datos_TFG_outliers/imputed_by_station/
    - Datos_TFG_outliers/imputed_global/
Salida:
    - Datos_TFG_outliers/encoded_ml/by_station/
    - Datos_TFG_outliers/encoded_ml/global/
    - Datos_TFG_outliers/encoded_dl/by_station/
    - Datos_TFG_outliers/encoded_dl/global/
    (dentro de cada carpeta, los archivos CSV con las transformaciones aplicadas)
    Además, en la carpeta encoded_dl se guardan archivos JSON con los mapeos de categorías.
"""

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("~/Documents/GitHub/TFGFinal/Datos_TFG_outliers")
INPUT_BY_STATION = os.path.join(BASE_DIR, "imputed_by_station")
INPUT_GLOBAL = os.path.join(BASE_DIR, "imputed_global")

OUTPUT_BASE = os.path.join(BASE_DIR, "encoded")
OUTPUT_ML_BY_STATION = os.path.join(OUTPUT_BASE, "ml", "by_station")
OUTPUT_ML_GLOBAL = os.path.join(OUTPUT_BASE, "ml", "global")
OUTPUT_DL_BY_STATION = os.path.join(OUTPUT_BASE, "dl", "by_station")
OUTPUT_DL_GLOBAL = os.path.join(OUTPUT_BASE, "dl", "global")

for path in [OUTPUT_ML_BY_STATION, OUTPUT_ML_GLOBAL, OUTPUT_DL_BY_STATION, OUTPUT_DL_GLOBAL]:
    os.makedirs(path, exist_ok=True)

# Columnas que se esperan en los archivos (además de las que se crearán)
# No todas pueden estar presentes; se manejarán con if.
ORIGINAL_NUM_COLS = ['NO', 'NO2', 'NOx', 'O3', 'Veloc.', 'Direc.', 'Temp.', 'R.Sol.', 'Dist.', 'Angulo']
CATEGORICAL_COLS = ['Estacion', 'Transecto']

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def cyclical_encode(series, period):
    """Convierte una serie numérica (con valores en [0, period)) a seno/coseno."""
    rad = 2 * np.pi * series / period
    sin = np.sin(rad)
    cos = np.cos(rad)
    return sin, cos

def add_datetime_features(df):
    """
    A partir del índice datetime, extrae:
    - hour (0-23)
    - dayofyear (1-366)
    - week (1-52)
    - month (1-12)
    - year (entero)
    y los transforma a seno/coseno (excepto year, que se deja lineal).
    Retorna df con nuevas columnas.
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("El índice debe ser DatetimeIndex")
    idx = df.index
    # Extraer componentes
    hour = idx.hour
    dayofyear = idx.dayofyear   # 1-366
    week = idx.isocalendar().week.astype(int)  # 1-53
    month = idx.month
    year = idx.year

    # Codificación cíclica
    df['hour_sin'], df['hour_cos'] = cyclical_encode(hour, 24)
    # dayofyear: periodo 365 (o 366, pero usamos 365 para simplificar)
    df['day_sin'], df['day_cos'] = cyclical_encode(dayofyear, 365)
    # week: periodo 52
    df['week_sin'], df['week_cos'] = cyclical_encode(week, 52)
    # month: periodo 12
    df['month_sin'], df['month_cos'] = cyclical_encode(month, 12)
    # año como entero (tendencia)
    df['year'] = year

    return df

def add_directional_features(df):
    """
    Convierte Direc. y Angulo (en grados) a seno/coseno.
    Asume que existen esas columnas. Las reemplaza por las nuevas.
    """
    for col in ['Direc.', 'Angulo']:
        if col in df.columns:
            rad = np.radians(df[col])
            df[f'{col}sin'] = np.sin(rad)
            df[f'{col}cos'] = np.cos(rad)
            # Eliminar columna original
            df.drop(columns=[col], inplace=True)
    return df

def encode_categorical_ml(df, cat_cols):
    """
    Aplica OneHotEncoder a las columnas categóricas.
    Retorna df con las dummies añadidas y elimina las originales.
    Nota: Usa pandas get_dummies por simplicidad; preserva el índice.
    """
    if not cat_cols:
        return df
    # Verificar qué columnas existen
    existing = [c for c in cat_cols if c in df.columns]
    if not existing:
        return df
    # One-hot con pandas get_dummies (genera columnas con prefijo)
    dummies = pd.get_dummies(df[existing], prefix=existing, dummy_na=False)
    df = pd.concat([df, dummies], axis=1)
    df.drop(columns=existing, inplace=True)
    return df

def encode_categorical_dl(df, cat_cols, save_mapping=True, mapping_file=None):
    """
    Convierte las columnas categóricas a enteros (0..n-1) usando factorize.
    Si save_mapping es True, guarda un dict con el mapeo (categoría -> código) en mapping_file (json).
    Retorna df con las columnas reemplazadas por enteros.
    """
    if not cat_cols:
        return df, {}
    existing = [c for c in cat_cols if c in df.columns]
    if not existing:
        return df, {}
    mapping = {}
    for col in existing:
        codes, uniques = pd.factorize(df[col])
        df[col] = codes
        mapping[col] = {str(k): int(v) for v, k in enumerate(uniques)}  # serializable
    if save_mapping and mapping_file:
        with open(mapping_file, 'w', encoding='utf-8') as f:
            json.dump(mapping, f, indent=2)
    return df, mapping

def process_file(input_path, output_dir_ml, output_dir_dl, is_global=False):
    """
    Procesa un archivo CSV: aplica codificaciones y guarda versiones ML y DL.
    input_path: ruta del archivo de entrada.
    output_dir_ml: carpeta donde guardar la versión ML (one-hot).
    output_dir_dl: carpeta donde guardar la versión DL (enteros).
    is_global: si es True, se asume que el archivo contiene datos de múltiples estaciones.
    """
    station_name = Path(input_path).stem
    print(f"Procesando: {station_name} (global={is_global})")

    # Cargar datos
    df = pd.read_csv(input_path, index_col=0, parse_dates=True)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)

    # Añadir características temporales
    df = add_datetime_features(df)

    # Añadir características direccionales (sin/cos)
    df = add_directional_features(df)

    # Hacer una copia para ML y otra para DL
    df_ml = df.copy()
    df_dl = df.copy()

    # Codificación ML: one-hot
    df_ml = encode_categorical_ml(df_ml, CATEGORICAL_COLS)

    # Codificación DL: factorize
    mapping_file = os.path.join(output_dir_dl, f"{station_name}_mapping.json")
    df_dl, _ = encode_categorical_dl(df_dl, CATEGORICAL_COLS, save_mapping=True, mapping_file=mapping_file)

    # Guardar archivos
    ml_path = os.path.join(output_dir_ml, f"{station_name}.csv")
    dl_path = os.path.join(output_dir_dl, f"{station_name}.csv")
    df_ml.to_csv(ml_path, index=True)
    df_dl.to_csv(dl_path, index=True)
    print(f"  ML guardado en {ml_path}")
    print(f"  DL guardado en {dl_path}")

def process_folder(input_folder, output_ml, output_dl, is_global=False):
    """Procesa todos los archivos CSV en input_folder."""
    files = list(Path(input_folder).glob("*.csv"))
    if not files:
        print(f"No se encontraron archivos CSV en {input_folder}")
        return
    for f in files:
        process_file(f, output_ml, output_dl, is_global)

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("="*60)
    print("Iniciando codificación de variables...")
    print("="*60)

    # Procesar datos por estación (individual)
    if os.path.exists(INPUT_BY_STATION):
        print("\n--- Datos por estación (individual) ---")
        process_folder(INPUT_BY_STATION, OUTPUT_ML_BY_STATION, OUTPUT_DL_BY_STATION, is_global=False)
    else:
        print(f"La carpeta {INPUT_BY_STATION} no existe. Se omite.")

    # Procesar datos globales (todas las estaciones juntas)
    if os.path.exists(INPUT_GLOBAL):
        print("\n--- Datos globales (todas las estaciones) ---")
        process_folder(INPUT_GLOBAL, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL, is_global=True)
    else:
        print(f"La carpeta {INPUT_GLOBAL} no existe. Se omite.")

    print("\nProceso completado. Revise las carpetas de salida.")
Notas importantes:
El código asume que los archivos de entrada tienen extensión .csv y que el índice es la primera columna (datetime). Si no es así, se puede ajustar.
Para la codificación cíclica de dayofyear se usa período 365. Si el dataset incluye años bisiestos, el día 366 se mapeará a un ángulo ligeramente diferente, pero es aceptable.
Para la codificación de week se usa período 52; en años con 53 semanas, la semana 53 también tendrá un ángulo, pero no hay problema.
Las columnas categóricas se factorizan con pd.factorize, que asigna enteros consecutivos en el orden de aparición. Eso es suficiente para embeddings.
Se guarda un archivo JSON por cada archivo procesado en la carpeta DL con los mapeos. En el caso global, el mapeo será único para todas las estaciones y transectos presentes en ese archivo.
Las columnas originales Direc. y Angulo se eliminan después de crear sus versiones seno/coseno.
Se conservan todas las demás columnas numéricas originales (NO, NO2, etc.) y el target O3.
Este código es autocontenido y debe ejecutarse después del Código 2.
python

#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 3: Codificación de variables temporales (datetime) en seno/coseno,
          transformación de dirección y ángulo a seno/coseno,
          y codificación de variables categóricas (Estacion, Transecto)
          para ML (OneHot) y DL (enteros para embedding).

Entrada: Carpetas con datos imputados:
    - Datos_TFG_outliers/imputed_by_station/
    - Datos_TFG_outliers/imputed_global/
Salida:
    - Datos_TFG_outliers/encoded_ml/by_station/
    - Datos_TFG_outliers/encoded_ml/global/
    - Datos_TFG_outliers/encoded_dl/by_station/
    - Datos_TFG_outliers/encoded_dl/global/
    (dentro de cada carpeta, los archivos CSV con las transformaciones aplicadas)
    Además, en la carpeta encoded_dl se guardan archivos JSON con los mapeos de categorías.
"""

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("~/Documents/GitHub/TFGFinal/Datos_TFG_outliers")
INPUT_BY_STATION = os.path.join(BASE_DIR, "imputed_by_station")
INPUT_GLOBAL = os.path.join(BASE_DIR, "imputed_global")

OUTPUT_BASE = os.path.join(BASE_DIR, "encoded")
OUTPUT_ML_BY_STATION = os.path.join(OUTPUT_BASE, "ml", "by_station")
OUTPUT_ML_GLOBAL = os.path.join(OUTPUT_BASE, "ml", "global")
OUTPUT_DL_BY_STATION = os.path.join(OUTPUT_BASE, "dl", "by_station")
OUTPUT_DL_GLOBAL = os.path.join(OUTPUT_BASE, "dl", "global")

for path in [OUTPUT_ML_BY_STATION, OUTPUT_ML_GLOBAL, OUTPUT_DL_BY_STATION, OUTPUT_DL_GLOBAL]:
    os.makedirs(path, exist_ok=True)

# Columnas que se esperan en los archivos (además de las que se crearán)
# No todas pueden estar presentes; se manejarán con if.
ORIGINAL_NUM_COLS = ['NO', 'NO2', 'NOx', 'O3', 'Veloc.', 'Direc.', 'Temp.', 'R.Sol.', 'Dist.', 'Angulo']
CATEGORICAL_COLS = ['Estacion', 'Transecto']

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def cyclical_encode(series, period):
    """Convierte una serie numérica (con valores en [0, period)) a seno/coseno."""
    rad = 2 * np.pi * series / period
    sin = np.sin(rad)
    cos = np.cos(rad)
    return sin, cos

def add_datetime_features(df):
    """
    A partir del índice datetime, extrae:
    - hour (0-23)
    - dayofyear (1-366)
    - week (1-52)
    - month (1-12)
    - year (entero)
    y los transforma a seno/coseno (excepto year, que se deja lineal).
    Retorna df con nuevas columnas.
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("El índice debe ser DatetimeIndex")
    idx = df.index
    # Extraer componentes
    hour = idx.hour
    dayofyear = idx.dayofyear   # 1-366
    week = idx.isocalendar().week.astype(int)  # 1-53
    month = idx.month
    year = idx.year

    # Codificación cíclica
    df['hour_sin'], df['hour_cos'] = cyclical_encode(hour, 24)
    # dayofyear: periodo 365 (o 366, pero usamos 365 para simplificar)
    df['day_sin'], df['day_cos'] = cyclical_encode(dayofyear, 365)
    # week: periodo 52
    df['week_sin'], df['week_cos'] = cyclical_encode(week, 52)
    # month: periodo 12
    df['month_sin'], df['month_cos'] = cyclical_encode(month, 12)
    # año como entero (tendencia)
    df['year'] = year

    return df

def add_directional_features(df):
    """
    Convierte Direc. y Angulo (en grados) a seno/coseno.
    Asume que existen esas columnas. Las reemplaza por las nuevas.
    """
    for col in ['Direc.', 'Angulo']:
        if col in df.columns:
            rad = np.radians(df[col])
            df[f'{col}sin'] = np.sin(rad)
            df[f'{col}cos'] = np.cos(rad)
            # Eliminar columna original
            df.drop(columns=[col], inplace=True)
    return df

def encode_categorical_ml(df, cat_cols):
    """
    Aplica OneHotEncoder a las columnas categóricas.
    Retorna df con las dummies añadidas y elimina las originales.
    Nota: Usa pandas get_dummies por simplicidad; preserva el índice.
    """
    if not cat_cols:
        return df
    # Verificar qué columnas existen
    existing = [c for c in cat_cols if c in df.columns]
    if not existing:
        return df
    # One-hot con pandas get_dummies (genera columnas con prefijo)
    dummies = pd.get_dummies(df[existing], prefix=existing, dummy_na=False)
    df = pd.concat([df, dummies], axis=1)
    df.drop(columns=existing, inplace=True)
    return df

def encode_categorical_dl(df, cat_cols, save_mapping=True, mapping_file=None):
    """
    Convierte las columnas categóricas a enteros (0..n-1) usando factorize.
    Si save_mapping es True, guarda un dict con el mapeo (categoría -> código) en mapping_file (json).
    Retorna df con las columnas reemplazadas por enteros.
    """
    if not cat_cols:
        return df, {}
    existing = [c for c in cat_cols if c in df.columns]
    if not existing:
        return df, {}
    mapping = {}
    for col in existing:
        codes, uniques = pd.factorize(df[col])
        df[col] = codes
        mapping[col] = {str(k): int(v) for v, k in enumerate(uniques)}  # serializable
    if save_mapping and mapping_file:
        with open(mapping_file, 'w', encoding='utf-8') as f:
            json.dump(mapping, f, indent=2)
    return df, mapping

def process_file(input_path, output_dir_ml, output_dir_dl, is_global=False):
    """
    Procesa un archivo CSV: aplica codificaciones y guarda versiones ML y DL.
    input_path: ruta del archivo de entrada.
    output_dir_ml: carpeta donde guardar la versión ML (one-hot).
    output_dir_dl: carpeta donde guardar la versión DL (enteros).
    is_global: si es True, se asume que el archivo contiene datos de múltiples estaciones.
    """
    station_name = Path(input_path).stem
    print(f"Procesando: {station_name} (global={is_global})")

    # Cargar datos
    df = pd.read_csv(input_path, index_col=0, parse_dates=True)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)

    # Añadir características temporales
    df = add_datetime_features(df)

    # Añadir características direccionales (sin/cos)
    df = add_directional_features(df)

    # Hacer una copia para ML y otra para DL
    df_ml = df.copy()
    df_dl = df.copy()

    # Codificación ML: one-hot
    df_ml = encode_categorical_ml(df_ml, CATEGORICAL_COLS)

    # Codificación DL: factorize
    mapping_file = os.path.join(output_dir_dl, f"{station_name}_mapping.json")
    df_dl, _ = encode_categorical_dl(df_dl, CATEGORICAL_COLS, save_mapping=True, mapping_file=mapping_file)

    # Guardar archivos
    ml_path = os.path.join(output_dir_ml, f"{station_name}.csv")
    dl_path = os.path.join(output_dir_dl, f"{station_name}.csv")
    df_ml.to_csv(ml_path, index=True)
    df_dl.to_csv(dl_path, index=True)
    print(f"  ML guardado en {ml_path}")
    print(f"  DL guardado en {dl_path}")

def process_folder(input_folder, output_ml, output_dl, is_global=False):
    """Procesa todos los archivos CSV en input_folder."""
    files = list(Path(input_folder).glob("*.csv"))
    if not files:
        print(f"No se encontraron archivos CSV en {input_folder}")
        return
    for f in files:
        process_file(f, output_ml, output_dl, is_global)

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("="*60)
    print("Iniciando codificación de variables...")
    print("="*60)

    # Procesar datos por estación (individual)
    if os.path.exists(INPUT_BY_STATION):
        print("\n--- Datos por estación (individual) ---")
        process_folder(INPUT_BY_STATION, OUTPUT_ML_BY_STATION, OUTPUT_DL_BY_STATION, is_global=False)
    else:
        print(f"La carpeta {INPUT_BY_STATION} no existe. Se omite.")

    # Procesar datos globales (todas las estaciones juntas)
    if os.path.exists(INPUT_GLOBAL):
        print("\n--- Datos globales (todas las estaciones) ---")
        process_folder(INPUT_GLOBAL, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL, is_global=True)
    else:
        print(f"La carpeta {INPUT_GLOBAL} no existe. Se omite.")

    print("\nProceso completado. Revise las carpetas de salida.")
Ahora creame el siguiente código siguiendo el orden que me recomendaste COMPLETO listo para copiar pegar y ejecutar
Ahora debemos generar el Código 4: Normalización según modelo y según método de análisis. Sin embargo, según el orden revisado que acordamos, después de la codificación (Código 3) sigue la creación de ventanas deslizantes (Código 4) y luego la partición (Código 5) y luego la normalización (Código 6). Pero el usuario pide: "Código 4: Normalización según modelo y según métodos de analisis". Debemos seguir su numeración original pero con el orden corregido internamente. En la respuesta anterior le sugerí un orden revisado, y él ha aceptado y está pidiendo el siguiente código. Debemos generar el código que corresponda al siguiente paso en ese orden revisado, que sería el de creación de ventanas deslizantes. Sin embargo, su solicitud textual es: "Ahora creame el siguiente código siguiendo el orden que me recomendaste COMPLETO listo para copiar pegar y ejecutar". Es decir, quiere el siguiente código en la secuencia que le recomendé, que después del Código 3 (codificación) sería el Código 4: Creación de ventanas deslizantes de 72h.

Pero cuidado: en su lista original el Código 4 era "Normalización según modelo y según métodos de analisis", y luego Código 5 "Creacionistas de ventanas deslizantes de 72h". En mi recomendación cambié el orden: primero ventanas, luego partición, luego normalización. Entonces, para ser coherentes con su petición de "siguiendo el orden que me recomendaste", el próximo código debería ser el de creación de ventanas deslizantes (lo que él llamaba Código 5 pero ahora sería el 4 en el nuevo orden). Sin embargo, como él ha numerado los códigos del 1 al 10, y yo sugerí un orden diferente, pero manteniendo los mismos números? No, en mi respuesta anterior di una lista con pasos renombrados, pero no reasigné números. El usuario ahora dice "el siguiente código" refiriéndose al siguiente en la secuencia que le recomendé. Entonces debemos interpretar que después del Código 3 (codificación) viene el siguiente paso: creación de ventanas deslizantes. Aunque en su numeración original eso sería el Código 5, él quiere que se lo entregue como el "siguiente código" en el orden recomendado.

Por lo tanto, debo generar el código para la creación de ventanas deslizantes de 72 horas (input 72 horas, output 72 horas) para cada estación, tanto para los datos por estación como para los globales. Además, debemos considerar que luego vendrá la partición y normalización. Pero este código debe ser completo y listo para ejecutar.

Además, recordar que el usuario especificó:

Ventana de entrada: 72 horas
Ventana de salida: 72 horas (multi-step directo)
Quiere un único modelo que prediga los 72 valores a la vez, es decir, para ML (RF, XGBoost) necesitaremos aplanar las 72 horas de entrada en features y tener 72 targets. Para RNN/LSTM, mantendremos la estructura 3D (muestras, timesteps, features). Por tanto, el código de ventanas debe generar dos tipos de datasets: uno para ML (tabla con 72*num_features columnas de entrada y 72 columnas de salida) y otro para DL (arrays 3D). Pero como aún no hemos separado train/valid/test, debemos crear las ventanas sobre la serie completa y luego partir. Otra opción es primero partir y luego crear ventanas, pero es más seguro crear ventanas después de tener la serie completa para evitar mezclar información futura en las ventanas (aunque si las ventanas se crean deslizando hacia adelante, no hay leakage porque cada ventana solo contiene pasado). La partición luego se hará sobre el conjunto de ventanas, manteniendo el orden temporal. Es más común crear ventanas primero (sobre la serie temporal completa) y luego dividir el conjunto de ventanas en train/validation/test respetando el orden (primeras ventanas para train, siguientes para valid, últimas para test). Esto evita que una ventana contenga datos futuros. Por tanto, haremos así.
Entonces, el código 4 (según orden recomendado) debe:

Leer los archivos codificados (de las carpetas ml y dl) para cada estación y para el global.
Para cada archivo, generar ventanas deslizantes de 72 horas de entrada y 72 horas de salida (target O3). La variable objetivo es O3, que ya tenemos.
Para los modelos de ML, necesitaremos convertir cada ventana en una fila con features aplanadas (72 * num_features) y 72 targets (valores futuros de O3).
Para los modelos de DL, mantendremos arrays 3D: (n_windows, 72, num_features) para X, y (n_windows, 72) para y (O3).
Guardar estos datasets en formato adecuado para su uso posterior. Podemos guardar en archivos .npy (para DL) y .csv (para ML) o en formato pickle.
Pero además, debemos considerar que hay dos metodologías: por estación individual y global. Para la global, tendremos un único archivo con todas las estaciones, pero las ventanas deben conservar la información de la estación (ya incluida como característica). Entonces, al crear ventanas para el dataset global, incluimos la estación como feature (ya sea one-hot o código). En el caso de ML, la estación ya estará en las columnas; en DL, será una feature más.

Además, el usuario especificó que para DL quiere usar embeddings para las categóricas. En el código anterior ya codificamos las categóricas como enteros para DL, y como one-hot para ML. Entonces, al crear ventanas para DL, esas columnas categóricas (Estacion, Transecto) ya son números enteros (0..n-1) que se usarán como índices para embedding. Pero hay que tener en cuenta que en la ventana deslizante, la estación no cambia dentro de una misma ventana (es constante para una estación), pero en el dataset global, al tener varias estaciones, la columna de estación variará entre ventanas. Eso está bien.

Por tanto, el código debería:

Definir función para crear ventanas dado un dataframe con índice temporal y columna O3 (target). Las features serán todas las demás columnas numéricas (incluyendo las codificadas) excepto O3? Cuidado: O3 también puede ser feature en el pasado? Normalmente para predecir O3 futuro, incluimos valores pasados de O3 y otras variables. Así que las features incluyen el O3 pasado también. Así que la columna O3 debe estar entre las features (para los rezagos). Pero también queremos predecir el O3 futuro, que será la salida. Por tanto, en la ventana de entrada incluimos los valores de O3 de las últimas 72 horas, y en la salida los valores de O3 de las siguientes 72 horas. Entonces, las features son todas las variables (incluyendo O3) en los 72 pasos pasados. La salida es solo O3 en los 72 pasos futuros.
La función debe devolver X (ventanas de entrada) e y (ventanas de salida) en formato para ML (2D) y DL (3D). Para ML, X será (n_windows, 72 * n_features) e y será (n_windows, 72). Para DL, X será (n_windows, 72, n_features) e y será (n_windows, 72).
Luego guardaremos estos arrays en archivos .npy y también guardaremos un CSV con las ventanas para ML? Podríamos guardar X_ml.csv y y_ml.csv, pero son grandes. Mejor guardar en formato .npy que es más eficiente. Pero para mantener compatibilidad, podemos guardar en .npz o en .npy.
También necesitaremos guardar la información de qué columnas son features y el orden. Podemos guardar un archivo features.json con la lista de nombres de las columnas (para luego interpretar).
Como hay múltiples estaciones y dos metodologías, generaremos carpetas organizadas.
Voy a diseñar el código.

Estructura de salida sugerida:
BASE_DIR = ~/Documents/GitHub/TFGFinal/Datos_TFG_outliers/windows/
by_station/
ml/
station1_X.npy, station1_y.npy, station1_features.json
station2_X.npy, ...
dl/
station1_X.npy, station1_y.npy, station1_features.json
...
global/
ml/
global_X.npy, global_y.npy, global_features.json
dl/
global_X.npy, global_y.npy, global_features.json

Pero cuidado: en el caso global, tenemos un único conjunto de ventanas que combina todas las estaciones. Las ventanas se crean a partir del dataframe concatenado, manteniendo el orden temporal global. Sin embargo, al tener varias estaciones, el dataframe concatenado no es una serie temporal continua porque hay saltos de una estación a otra. Pero si hemos concatenado simplemente una debajo de otra, el índice temporal se mezcla. Para evitar ventanas que crucen entre estaciones, debemos asegurarnos de que las ventanas solo se creen dentro de cada estación por separado y luego concatenar las ventanas. Es decir, para el dataset global, primero creamos ventanas para cada estación individualmente y luego concatenamos todas las ventanas en un solo conjunto. De esta forma, cada ventana pertenece a una sola estación y no hay mezcla temporal entre estaciones. La variable de estación ya está presente en las features, por lo que el modelo podrá aprender diferencias entre estaciones. Esta es la forma correcta.

Por tanto, para el caso global, procesaremos cada estación por separado generando sus ventanas, y luego concatenaremos todas las ventanas en un único array X e y. Además, necesitaremos guardar un mapeo de qué ventana pertenece a qué estación (por si acaso), pero no es esencial.

Implementaremos una función create_windows(df, target_col='O3', window_in=72, window_out=72) que devuelva X_ml, y_ml, X_dl, y_dl. Supondremos que el dataframe tiene todas las features (incluyendo las codificadas) y que no contiene NaN (ya imputado). El índice es datetime y está ordenado. La función debe recorrer el dataframe y extraer ventanas que no se solapen? Normalmente se usan ventanas deslizantes con paso 1 (cada hora). Así que generaremos una ventana por cada instante donde haya suficientes datos hacia atrás y hacia adelante. Es decir, para un instante t, la entrada son los valores de t-71 a t (72 horas) y la salida son t+1 a t+72. Pero si queremos que la salida sean las 72 horas siguientes, entonces para cada t desde el índice 71 hasta len(df)-72. Pero cuidado: si usamos paso 1, habrá superposición entre ventanas, lo cual es común en series temporales para aumentar el número de muestras. Esto está bien siempre que luego en la partición no mezclemos ventanas adyacentes en diferentes conjuntos (para evitar leakage), pero si hacemos partición cronológica estricta (primeras ventanas train, siguientes valid, últimas test), no hay problema porque las ventanas adyacentes están en el mismo conjunto. Así que usaremos paso 1.

Entonces, definimos:

n = len(df)
X_ml_list = []
y_ml_list = []
X_dl_list = []
y_dl_list = []

for i in range(window_in, n - window_out):

entrada: i-window_in : i

in_data = df.iloc[i-window_in:i].values # shape (window_in, n_features)

salida: i : i+window_out (target O3)

out_data = df[target_col].iloc[i:i+window_out].values # shape (window_out,)
X_dl_list.append(in_data)
y_dl_list.append(out_data)

para ML, aplanamos in_data

X_ml_list.append(in_data.flatten())
y_ml_list.append(out_data)

Luego convertimos a arrays numpy.

Nota: df.values incluye todas las columnas, incluida O3. Pero en la salida solo queremos O3, así que tomamos df[target_col]. Esto es correcto.

Para el caso global, haremos esto para cada estación y luego concatenamos las listas.

También necesitamos guardar la lista de nombres de columnas (features) para poder interpretar los arrays más adelante. Para ML, el orden de las features aplanadas será: [feature1_t-71, feature2_t-71, ..., featureN_t-71, feature1_t-70, ...] etc. Para DL, es más fácil porque mantenemos la estructura. Guardaremos features.json con la lista de nombres de columnas.

Ahora, escribamos el código completo.
python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 4 (orden recomendado): Creación de ventanas deslizantes de 72h.
Entrada: datos codificados (salida del Código 3) en las carpetas:
    - encoded/ml/by_station/
    - encoded/ml/global/
    - encoded/dl/by_station/
    - encoded/dl/global/
Salida: ventanas en formato numpy arrays (.npy) para modelos ML (2D) y DL (3D),
        además de archivos JSON con la lista de features.
Estructura de salida:
    windows/
        by_station/
            ml/
                estacion1_X.npy, estacion1_y.npy, features.json
                estacion2_X.npy, estacion2_y.npy, features.json
                ...
            dl/
                estacion1_X.npy, estacion1_y.npy, features.json
                ...
        global/
            ml/
                global_X.npy, global_y.npy, features.json   (ventanas combinadas)
            dl/
                global_X.npy, global_y.npy, features.json
"""

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("~/Documents/GitHub/TFGFinal/Datos_TFG_outliers")
INPUT_ML_BY_STATION = os.path.join(BASE_DIR, "encoded", "ml", "by_station")
INPUT_ML_GLOBAL = os.path.join(BASE_DIR, "encoded", "ml", "global")
INPUT_DL_BY_STATION = os.path.join(BASE_DIR, "encoded", "dl", "by_station")
INPUT_DL_GLOBAL = os.path.join(BASE_DIR, "encoded", "dl", "global")

OUTPUT_WINDOWS = os.path.join(BASE_DIR, "windows")
OUTPUT_ML_BY_STATION = os.path.join(OUTPUT_WINDOWS, "by_station", "ml")
OUTPUT_DL_BY_STATION = os.path.join(OUTPUT_WINDOWS, "by_station", "dl")
OUTPUT_ML_GLOBAL = os.path.join(OUTPUT_WINDOWS, "global", "ml")
OUTPUT_DL_GLOBAL = os.path.join(OUTPUT_WINDOWS, "global", "dl")

for path in [OUTPUT_ML_BY_STATION, OUTPUT_DL_BY_STATION, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL]:
    os.makedirs(path, exist_ok=True)

# Parámetros de ventanas
WINDOW_IN = 72      # horas de entrada
WINDOW_OUT = 72     # horas de salida (predicción)
TARGET_COL = 'O3'   # nombre de la columna objetivo

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def create_windows_from_df(df, target_col=TARGET_COL, window_in=WINDOW_IN, window_out=WINDOW_OUT):
    """
    Genera ventanas deslizantes a partir de un DataFrame con índice temporal ordenado.
    Retorna:
        X_ml: (n_windows, window_in * n_features)  (aplanado)
        y_ml: (n_windows, window_out)
        X_dl: (n_windows, window_in, n_features)
        y_dl: (n_windows, window_out)
        feature_names: lista de nombres de las columnas (orden original)
    """
    if not isinstance(df.index, pd.DatetimeIndex):
        raise ValueError("El DataFrame debe tener índice DatetimeIndex")
    df_sorted = df.sort_index()  # por si acaso
    data = df_sorted.values
    n_features = data.shape[1]
    feature_names = df_sorted.columns.tolist()

    n = len(df_sorted)
    X_ml_list = []
    y_ml_list = []
    X_dl_list = []
    y_dl_list = []

    for i in range(window_in, n - window_out):
        # Ventana de entrada: [i-window_in , i)
        in_data = data[i - window_in:i, :]                # shape (window_in, n_features)
        # Ventana de salida (solo target): [i , i+window_out)
        out_data = df_sorted[target_col].iloc[i:i + window_out].values  # shape (window_out,)
        X_dl_list.append(in_data)
        y_dl_list.append(out_data)
        X_ml_list.append(in_data.flatten())   # aplanado: (window_in * n_features,)
        y_ml_list.append(out_data)

    if len(X_ml_list) == 0:
        print("  Aviso: no se generaron ventanas (serie demasiado corta).")
        return None, None, None, None, feature_names

    X_ml = np.array(X_ml_list, dtype=np.float32)
    y_ml = np.array(y_ml_list, dtype=np.float32)
    X_dl = np.array(X_dl_list, dtype=np.float32)
    y_dl = np.array(y_dl_list, dtype=np.float32)

    return X_ml, y_ml, X_dl, y_dl, feature_names

def process_station_file(file_path, output_ml_dir, output_dl_dir):
    """
    Procesa un archivo CSV de una estación (ya codificado) y guarda las ventanas.
    """
    station_name = Path(file_path).stem
    print(f"  Procesando estación: {station_name}")

    df = pd.read_csv(file_path, index_col=0, parse_dates=True)
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index)

    # Crear ventanas
    result = create_windows_from_df(df, target_col=TARGET_COL,
                                    window_in=WINDOW_IN, window_out=WINDOW_OUT)
    if result[0] is None:
        print(f"    No se generaron ventanas para {station_name}")
        return
    X_ml, y_ml, X_dl, y_dl, feature_names = result

    # Guardar archivos ML
    np.save(os.path.join(output_ml_dir, f"{station_name}_X.npy"), X_ml)
    np.save(os.path.join(output_ml_dir, f"{station_name}_y.npy"), y_ml)
    with open(os.path.join(output_ml_dir, f"{station_name}_features.json"), 'w') as f:
        json.dump(feature_names, f, indent=2)

    # Guardar archivos DL
    np.save(os.path.join(output_dl_dir, f"{station_name}_X.npy"), X_dl)
    np.save(os.path.join(output_dl_dir, f"{station_name}_y.npy"), y_dl)
    with open(os.path.join(output_dl_dir, f"{station_name}_features.json"), 'w') as f:
        json.dump(feature_names, f, indent=2)

    print(f"    Ventanas guardadas: {len(X_ml)} muestras")

def process_global_folder(input_dir_ml, input_dir_dl, output_ml_dir, output_dl_dir):
    """
    Para el caso global: se toman los archivos de todas las estaciones (de las carpetas ml y dl),
    se generan ventanas por separado y se concatenan en un único conjunto.
    input_dir_ml: carpeta con archivos CSV para ML (one-hot)
    input_dir_dl: carpeta con archivos CSV para DL (enteros)
    Ambas deben contener los mismos nombres de estación (uno por estación).
    """
    # Buscar archivos CSV en input_dir_ml (usamos los nombres para iterar)
    ml_files = list(Path(input_dir_ml).glob("*.csv"))
    if not ml_files:
        print(f"  No se encontraron archivos CSV en {input_dir_ml}")
        return

    all_X_ml = []
    all_y_ml = []
    all_X_dl = []
    all_y_dl = []
    feature_names_ml = None
    feature_names_dl = None

    for ml_file in ml_files:
        station_name = ml_file.stem
        dl_file = os.path.join(input_dir_dl, f"{station_name}.csv")
        if not os.path.exists(dl_file):
            print(f"  Aviso: no existe el archivo DL para {station_name}, se omite.")
            continue

        # Cargar ML
        df_ml = pd.read_csv(ml_file, index_col=0, parse_dates=True)
        # Cargar DL
        df_dl = pd.read_csv(dl_file, index_col=0, parse_dates=True)

        # Verificar que los índices sean comparables
        # (deberían ser iguales porque provienen de la misma estación)
        if not df_ml.index.equals(df_dl.index):
            print(f"  Aviso: los índices de {station_name} no coinciden entre ML y DL. Se usará ML para features.")
            # En todo caso, usamos el de ML para consistencia
            df_dl = df_dl.reindex(df_ml.index)

        # Generar ventanas para ML (usando df_ml)
        res_ml = create_windows_from_df(df_ml, target_col=TARGET_COL,
                                        window_in=WINDOW_IN, window_out=WINDOW_OUT)
        if res_ml[0] is None:
            continue
        X_ml, y_ml, _, _, feat_ml = res_ml
        if feature_names_ml is None:
            feature_names_ml = feat_ml
        else:
            if feat_ml != feature_names_ml:
                print(f"  Error: las features de {station_name} no coinciden con las anteriores. Se omitirá.")
                continue
        all_X_ml.append(X_ml)
        all_y_ml.append(y_ml)

        # Generar ventanas para DL (usando df_dl)
        res_dl = create_windows_from_df(df_dl, target_col=TARGET_COL,
                                        window_in=WINDOW_IN, window_out=WINDOW_OUT)
        if res_dl[0] is None:
            continue
        X_dl, y_dl, _, _, feat_dl = res_dl
        if feature_names_dl is None:
            feature_names_dl = feat_dl
        else:
            if feat_dl != feature_names_dl:
                print(f"  Error: las features DL de {station_name} no coinciden.")
                continue
        all_X_dl.append(X_dl)
        all_y_dl.append(y_dl)

    if not all_X_ml:
        print("  No se generaron ventanas para ninguna estación.")
        return

    # Concatenar
    X_ml_global = np.concatenate(all_X_ml, axis=0)
    y_ml_global = np.concatenate(all_y_ml, axis=0)
    X_dl_global = np.concatenate(all_X_dl, axis=0)
    y_dl_global = np.concatenate(all_y_dl, axis=0)

    print(f"  Total ventanas globales ML: {X_ml_global.shape[0]}")
    print(f"  Total ventanas globales DL: {X_dl_global.shape[0]}")

    # Guardar
    np.save(os.path.join(output_ml_dir, "global_X.npy"), X_ml_global)
    np.save(os.path.join(output_ml_dir, "global_y.npy"), y_ml_global)
    with open(os.path.join(output_ml_dir, "global_features.json"), 'w') as f:
        json.dump(feature_names_ml, f, indent=2)

    np.save(os.path.join(output_dl_dir, "global_X.npy"), X_dl_global)
    np.save(os.path.join(output_dl_dir, "global_y.npy"), y_dl_global)
    with open(os.path.join(output_dl_dir, "global_features.json"), 'w') as f:
        json.dump(feature_names_dl, f, indent=2)

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("=" * 60)
    print("Creación de ventanas deslizantes (72h in / 72h out)")
    print("=" * 60)

    # 1. Procesar por estación (individual)
    print("\n--- Procesando datos por estación (individual) ---")
    if os.path.exists(INPUT_ML_BY_STATION) and os.path.exists(INPUT_DL_BY_STATION):
        ml_files = list(Path(INPUT_ML_BY_STATION).glob("*.csv"))
        dl_files = list(Path(INPUT_DL_BY_STATION).glob("*.csv"))
        # Intersectar nombres
        ml_names = {f.stem for f in ml_files}
        dl_names = {f.stem for f in dl_files}
        common = ml_names.intersection(dl_names)
        for station in common:
            ml_path = os.path.join(INPUT_ML_BY_STATION, f"{station}.csv")
            dl_path = os.path.join(INPUT_DL_BY_STATION, f"{station}.csv")
            # Para ML usamos el archivo ML, para DL el archivo DL
            # process_station_file espera un archivo y dos directorios de salida,
            # pero aquí tenemos dos archivos distintos. Vamos a procesar por separado:
            # Primero ML
            df_ml = pd.read_csv(ml_path, index_col=0, parse_dates=True)
            res_ml = create_windows_from_df(df_ml)
            if res_ml[0] is not None:
                X_ml, y_ml, _, _, feat_ml = res_ml
                np.save(os.path.join(OUTPUT_ML_BY_STATION, f"{station}_X.npy"), X_ml)
                np.save(os.path.join(OUTPUT_ML_BY_STATION, f"{station}_y.npy"), y_ml)
                with open(os.path.join(OUTPUT_ML_BY_STATION, f"{station}_features.json"), 'w') as f:
                    json.dump(feat_ml, f)
                print(f"  ML {station}: {len(X_ml)} ventanas")
            # Luego DL
            df_dl = pd.read_csv(dl_path, index_col=0, parse_dates=True)
            res_dl = create_windows_from_df(df_dl)
            if res_dl[0] is not None:
                X_dl, y_dl, _, _, feat_dl = res_dl
                np.save(os.path.join(OUTPUT_DL_BY_STATION, f"{station}_X.npy"), X_dl)
                np.save(os.path.join(OUTPUT_DL_BY_STATION, f"{station}_y.npy"), y_dl)
                with open(os.path.join(OUTPUT_DL_BY_STATION, f"{station}_features.json"), 'w') as f:
                    json.dump(feat_dl, f)
                print(f"  DL {station}: {len(X_dl)} ventanas")
    else:
        print("  Las carpetas de entrada por estación no existen. Se omite.")

    # 2. Procesar global (todas las estaciones juntas)
    print("\n--- Procesando datos globales ---")
    if os.path.exists(INPUT_ML_GLOBAL) and os.path.exists(INPUT_DL_GLOBAL):
        process_global_folder(INPUT_ML_GLOBAL, INPUT_DL_GLOBAL,
                              OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL)
    else:
        print("  Las carpetas de entrada global no existen. Se omite.")

    print("\nProceso completado. Revise la carpeta:")
    print(f"  {OUTPUT_WINDOWS}")
Ahora creame el siguiente código siguiendo el orden que me recomendaste COMPLETO listo para copiar pegar y ejecutar
Ahora el usuario pide el siguiente código en el orden recomendado. Hasta ahora hemos hecho:

Código 1: outliers
Código 2: imputación
Código 3: codificación (datetime, dirección, categóricas)
Código 4: ventanas deslizantes (según el orden revisado, este paso va antes de la normalización y partición)
El siguiente paso en el orden revisado sería: Partición de datos (train/validation/test) y normalización (solo para modelos que la requieran). Pero el usuario dijo "Ahora creame el siguiente código siguiendo el orden que me recomendaste COMPLETO listo para copiar pegar y ejecutar". Debemos recordar cuál es el siguiente paso según la lista que acordamos:

Orden revisado que propuse:
Tratamiento de outliers (Código 1)
Imputación de valores NaN (Código 2)
Creación de variables temporales cíclicas y transformación de ángulo (Código 3)
Codificación de variables categóricas (también en Código 3, ya lo hicimos)
Creación de ventanas deslizantes (Código 4)
Partición de las ventanas en train/validation/test de forma cronológica
Normalización (solo para modelos que la requieran)
Preparación de datasets específicos por modelo (individual y global)
Entrenamiento, búsqueda de hiperparámetros y evaluación para cada modelo (RF, XGBoost, RNN, LSTM)
Por tanto, el siguiente código (Código 5) debería ser la partición de datos y normalización. Además, debemos tener en cuenta que la normalización debe hacerse después de la partición para evitar leakage. También debemos considerar que para árboles no se normaliza, pero para redes sí. Así que el código debe generar conjuntos de entrenamiento, validación y prueba normalizados para los modelos que lo necesiten, y sin normalizar para los que no.

El usuario ya ha dado especificaciones sobre la partición: 70% train, 15% validation, 15% test, con fechas concretas: train hasta 2023-12-31, validation 2024-01-01 a 2024-12-31, test 2025-01-01 a 2025-12-31. También quiere validación cruzada TimeSeriesSplit para búsqueda de hiperparámetros, pero eso se hará dentro de cada modelo. En este paso solo necesitamos dividir las ventanas en conjuntos fijos según esas fechas.

Además, tenemos dos estrategias: por estación individual y global. En el caso global, las ventanas ya están concatenadas, pero debemos asegurarnos de que la partición respeta el orden temporal global (no mezclar ventanas de diferentes tiempos). En el caso global, las ventanas provienen de todas las estaciones, pero el tiempo es secuencial (cada ventana tiene un timestamp asociado al inicio). Necesitamos poder asignar cada ventana a train/val/test según su fecha de inicio. Para ello, al crear las ventanas deberíamos haber guardado también los timestamps. En el código anterior no guardamos los timestamps. Para poder hacer la partición temporal necesitamos saber a qué fecha corresponde cada ventana. Por tanto, debemos modificar el código de creación de ventanas para que también guarde los índices temporales asociados a cada ventana. O bien, en este paso, podemos leer los datos originales y generar las ventanas nuevamente junto con los timestamps. Pero eso sería ineficiente. Lo mejor es que en el código de ventanas hayamos guardado también un array con los timestamps de inicio de cada ventana. Como no lo hicimos, tendremos que regenerar las ventanas en este código o leer los datos originales y generar las ventanas sobre la marcha. Para mantener la modularidad, es preferible que cada código sea autocontenido y no dependa de que el anterior haya guardado algo que no guardó. Pero dado que ya tenemos los archivos de ventanas (X e y), pero no los timestamps, no podemos hacer la partición temporal basada en fechas a menos que también tengamos los índices. Por lo tanto, en este nuevo código deberemos leer los datos originales (los CSV codificados) y volver a generar las ventanas, pero esta vez preservando los timestamps. O bien, leer los archivos de ventanas y también leer los índices de los datos originales para asignar fechas a cada ventana (lo cual es posible si sabemos la posición de inicio de cada ventana, pero eso requeriría haber guardado los índices).

Para simplificar y garantizar que la partición sea correcta, propongo que en este código leamos los CSV de las carpetas encoded (ml y dl) y generemos las ventanas nuevamente, pero además guardemos los timestamps de inicio. Luego, con esos timestamps, aplicamos la partición por fechas. Esto asegura que tenemos control total. Aunque es un poco redundante, es seguro.

Alternativamente, podríamos leer los archivos de ventanas (X.npy) y también los CSV originales para recuperar las fechas, pero es más complicado.

Por tanto, el Código 5 hará:

Para cada estación (individual):
Cargar el CSV codificado (desde la carpeta ml y dl correspondiente).
Generar ventanas (igual que antes) pero además devolver los índices (fechas de inicio de cada ventana).
Dividir las ventanas en train, val, test según las fechas dadas.
Aplicar normalización (MinMaxScaler) a las ventanas de entrenamiento (solo para DL) y transformar val y test con esos escaladores.
Guardar los conjuntos en carpetas separadas (train/val/test) para cada modelo (RF, XGBoost no necesitan normalización, pero para ellos guardaremos los datos sin normalizar; para DL guardaremos normalizados).
También guardar los escaladores para poder invertir la normalización del target si es necesario.
Para el caso global:
Similar, pero concatenando todas las estaciones. La partición temporal debe hacerse globalmente: todas las ventanas de todas las estaciones se mezclan, pero cada ventana tiene su fecha. El conjunto de train será todas las ventanas con fecha <= 2023-12-31, validation 2024, test 2025. Esto es correcto porque no hay orden entre estaciones, solo el tiempo.
Además, debemos considerar que para los modelos de árbol (RF, XGBoost) usaremos los datos sin normalizar (o con normalización si se quiere, pero no es necesaria). El usuario dijo: "Para redes neuronales quiero MinMaxScaler" y "Aplica la normalización a todas las características numéricas incluyendo seno y coseno". También dijo "si es posible y coherente no escalar O3 prefiero que no se escale, pero si no es coherente no escalarlo entonces prefiero que se escale." Esto implica que para redes, probablemente queremos escalar todo, incluyendo O3, pero luego tendremos que desescalar las predicciones para calcular métricas. Es más coherente escalar el target también porque la red suele funcionar mejor con entradas y salidas en rangos similares. Así que escalaremos O3 también, y luego desescalaremos.

Para árboles, no escalamos nada.

Por tanto, en este código generaremos dos versiones de cada conjunto: una sin escalar (para árboles) y otra escalada con MinMaxScaler (para redes). Además, guardaremos los escaladores (para poder desescalar después).

También debemos considerar que la variable objetivo O3 está en las ventanas y en los features. Al escalar, escalamos todo el conjunto de features (X) y también el target (y). Pero para árboles, no se escala.

Estructura de salida propuesta:

text
windows_partitioned/
    by_station/
        ml/  (datos sin normalizar para árboles)
            station1/
                train_X.npy, train_y.npy
                val_X.npy, val_y.npy
                test_X.npy, test_y.npy
                features.json
            station2/
                ...
        dl/  (datos normalizados para redes)
            station1/
                train_X.npy, train_y.npy
                val_X.npy, val_y.npy
                test_X.npy, test_y.npy
                features.json
                scaler_X.pkl, scaler_y.pkl   (guardamos los escaladores)
    global/
        ml/
            train_X.npy, train_y.npy
            val_X.npy, val_y.npy
            test_X.npy, test_y.npy
            features.json
        dl/
            train_X.npy, train_y.npy
            val_X.npy, val_y.npy
            test_X.npy, test_y.npy
            features.json
            scaler_X.pkl, scaler_y.pkl
Usaremos pickle para guardar los escaladores.

Ahora, redactemos el código.
python
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 5 (orden recomendado): Partición temporal (train/validation/test) y normalización.
Entrada: datos codificados (salida del Código 3) en las carpetas:
    - encoded/ml/by_station/
    - encoded/ml/global/
    - encoded/dl/by_station/
    - encoded/dl/global/
Salida: conjuntos particionados y normalizados (para DL) en:
    windows_partitioned/
        by_station/
            ml/ (sin normalizar) y dl/ (normalizados)
        global/
            ml/ y dl/
Se guardan también los escaladores para poder invertir la normalización.
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("~/Documents/GitHub/TFGFinal/Datos_TFG_outliers")
INPUT_ML_BY_STATION = os.path.join(BASE_DIR, "encoded", "ml", "by_station")
INPUT_ML_GLOBAL = os.path.join(BASE_DIR, "encoded", "ml", "global")
INPUT_DL_BY_STATION = os.path.join(BASE_DIR, "encoded", "dl", "by_station")
INPUT_DL_GLOBAL = os.path.join(BASE_DIR, "encoded", "dl", "global")

OUTPUT_DIR = os.path.join(BASE_DIR, "windows_partitioned")
OUTPUT_ML_BY_STATION = os.path.join(OUTPUT_DIR, "by_station", "ml")
OUTPUT_DL_BY_STATION = os.path.join(OUTPUT_DIR, "by_station", "dl")
OUTPUT_ML_GLOBAL = os.path.join(OUTPUT_DIR, "global", "ml")
OUTPUT_DL_GLOBAL = os.path.join(OUTPUT_DIR, "global", "dl")

for path in [OUTPUT_ML_BY_STATION, OUTPUT_DL_BY_STATION, OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL]:
    os.makedirs(path, exist_ok=True)

# Parámetros de ventanas
WINDOW_IN = 72
WINDOW_OUT = 72
TARGET_COL = 'O3'

# Fechas de corte (según especificación)
TRAIN_END = '2023-12-31 23:00:00'
VAL_START = '2024-01-01 00:00:00'
VAL_END = '2024-12-31 23:00:00'
TEST_START = '2025-01-01 00:00:00'
# TEST_END = '2025-12-31 23:00:00'  # se asume hasta el final

def load_entity_data(entity_dir, entity_name):
    X_path = os.path.join(entity_dir, f"{entity_name}_X.npy")
    y_path = os.path.join(entity_dir, f"{entity_name}_y.npy")
    ts_path = os.path.join(entity_dir, f"{entity_name}_timestamps.npy")
    if not (os.path.exists(X_path) and os.path.exists(y_path) and os.path.exists(ts_path)):
        return None, None, None
    X = np.load(X_path)
    y = np.load(y_path)
    timestamps = np.load(ts_path)
    timestamps = pd.to_datetime(timestamps)
    return X, y, timestamps

def split_by_timestamps(X, y, timestamps):
    train_mask = timestamps <= pd.to_datetime(TRAIN_END)
    val_mask = (timestamps >= pd.to_datetime(VAL_START)) & (timestamps <= pd.to_datetime(VAL_END))
    test_mask = timestamps >= pd.to_datetime(TEST_START)
    return (X[train_mask], y[train_mask]), (X[val_mask], y[val_mask]), (X[test_mask], y[test_mask])

def normalize_data(X_train, X_val, X_test, y_train, y_val, y_test):
    n_train, win_in, n_feat = X_train.shape
    n_val = X_val.shape[0]
    n_test = X_test.shape[0]

    X_train_flat = X_train.reshape(-1, n_feat)
    X_val_flat = X_val.reshape(-1, n_feat)
    X_test_flat = X_test.reshape(-1, n_feat)

    scaler_X = MinMaxScaler()
    X_train_scaled_flat = scaler_X.fit_transform(X_train_flat)
    X_val_scaled_flat = scaler_X.transform(X_val_flat)
    X_test_scaled_flat = scaler_X.transform(X_test_flat)

    X_train_sc = X_train_scaled_flat.reshape(n_train, win_in, n_feat)
    X_val_sc = X_val_scaled_flat.reshape(n_val, win_in, n_feat)
    X_test_sc = X_test_scaled_flat.reshape(n_test, win_in, n_feat)

    scaler_y = MinMaxScaler()
    y_train_flat = y_train.reshape(-1, 1)
    y_val_flat = y_val.reshape(-1, 1)
    y_test_flat = y_test.reshape(-1, 1)
    scaler_y.fit(y_train_flat)
    y_train_sc = scaler_y.transform(y_train_flat).reshape(y_train.shape)
    y_val_sc = scaler_y.transform(y_val_flat).reshape(y_val.shape)
    y_test_sc = scaler_y.transform(y_test_flat).reshape(y_test.shape)

    return X_train_sc, X_val_sc, X_test_sc, y_train_sc, y_val_sc, y_test_sc, scaler_X, scaler_y

def save_split_ml(output_dir, entity_name, X_train, y_train, X_val, y_val, X_test, y_test):
    save_dir = os.path.join(output_dir, entity_name)
    os.makedirs(save_dir, exist_ok=True)
    np.save(os.path.join(save_dir, "train_X.npy"), X_train)
    np.save(os.path.join(save_dir, "train_y.npy"), y_train)
    np.save(os.path.join(save_dir, "val_X.npy"), X_val)
    np.save(os.path.join(save_dir, "val_y.npy"), y_val)
    np.save(os.path.join(save_dir, "test_X.npy"), X_test)
    np.save(os.path.join(save_dir, "test_y.npy"), y_test)

def save_split_dl(output_dir, entity_name, X_train, y_train, X_val, y_val, X_test, y_test,
                  scaler_X, scaler_y):
    save_dir = os.path.join(output_dir, entity_name)
    os.makedirs(save_dir, exist_ok=True)
    np.save(os.path.join(save_dir, "train_X.npy"), X_train)
    np.save(os.path.join(save_dir, "train_y.npy"), y_train)
    np.save(os.path.join(save_dir, "val_X.npy"), X_val)
    np.save(os.path.join(save_dir, "val_y.npy"), y_val)
    np.save(os.path.join(save_dir, "test_X.npy"), X_test)
    np.save(os.path.join(save_dir, "test_y.npy"), y_test)
    with open(os.path.join(save_dir, "scaler_X.pkl"), 'wb') as f:
        pickle.dump(scaler_X, f)
    with open(os.path.join(save_dir, "scaler_y.pkl"), 'wb') as f:
        pickle.dump(scaler_y, f)

def process_entity(entity_name, input_ml_dir, input_dl_dir, output_ml_dir, output_dl_dir):
    print(f"  Procesando {entity_name}...")
    X_ml, y_ml, ts_ml = load_entity_data(input_ml_dir, entity_name)
    if X_ml is None:
        return
    X_dl, y_dl, ts_dl = load_entity_data(input_dl_dir, entity_name)
    if X_dl is None:
        return
    if not np.array_equal(ts_ml, ts_dl):
        print(f"    Error: timestamps no coinciden. Se omite.")
        return

    (X_train_ml, y_train_ml), (X_val_ml, y_val_ml), (X_test_ml, y_test_ml) = split_by_timestamps(X_ml, y_ml, ts_ml)
    (X_train_dl, y_train_dl), (X_val_dl, y_val_dl), (X_test_dl, y_test_dl) = split_by_timestamps(X_dl, y_dl, ts_dl)

    save_split_ml(output_ml_dir, entity_name,
                  X_train_ml, y_train_ml, X_val_ml, y_val_ml, X_test_ml, y_test_ml)

    if len(X_train_dl) > 0:
        X_train_sc, X_val_sc, X_test_sc, y_train_sc, y_val_sc, y_test_sc, scaler_X, scaler_y = normalize_data(
            X_train_dl, X_val_dl, X_test_dl, y_train_dl, y_val_dl, y_test_dl
        )
        save_split_dl(output_dl_dir, entity_name,
                      X_train_sc, y_train_sc, X_val_sc, y_val_sc, X_test_sc, y_test_sc,
                      scaler_X, scaler_y)
    else:
        print(f"    No hay datos de entrenamiento para {entity_name}, se omite DL.")

def process_by_transect():
    print("\n--- Procesando datos por transecto ---")
    if not os.path.exists(INPUT_ML_TRANSECT) or not os.path.exists(INPUT_DL_TRANSECT):
        return
    ml_files = [f.stem.replace("_X", "") for f in Path(INPUT_ML_TRANSECT).glob("*_X.npy")]
    dl_files = [f.stem.replace("_X", "") for f in Path(INPUT_DL_TRANSECT).glob("*_X.npy")]
    common = set(ml_files) & set(dl_files)
    for entity in sorted(common):
        process_entity(entity, INPUT_ML_TRANSECT, INPUT_DL_TRANSECT,
                       OUTPUT_ML_TRANSECT, OUTPUT_DL_TRANSECT)

def process_global():
    print("\n--- Procesando datos globales (por estación) ---")
    if not os.path.exists(INPUT_ML_GLOBAL) or not os.path.exists(INPUT_DL_GLOBAL):
        return
    ml_files = [f.stem.replace("_X", "") for f in Path(INPUT_ML_GLOBAL).glob("*_X.npy")]
    dl_files = [f.stem.replace("_X", "") for f in Path(INPUT_DL_GLOBAL).glob("*_X.npy")]
    common = set(ml_files) & set(dl_files)
    for entity in sorted(common):
        process_entity(entity, INPUT_ML_GLOBAL, INPUT_DL_GLOBAL,
                       OUTPUT_ML_GLOBAL, OUTPUT_DL_GLOBAL)

if __name__ == "__main__":
    print("="*60)
    print("Partición temporal y normalización de ventanas")
    print("="*60)
    process_by_transect()
    process_global()
    print("\nProceso completado. Revise la carpeta:")
    print(f"  {OUTPUT_DIR}")

Partición temporal y normalización de ventanas

--- Procesando datos por transecto ---
  Procesando Transecto_1...
  Procesando Transecto_2...

--- Procesando datos globales (por estación) ---
  Procesando T1_E1_Alicante...
  Procesando T1_E2_Elda...
  Procesando T2_E1_Elche...
  Procesando T2_E2_Elda...

Proceso completado. Revise la carpeta:
  /Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/windows_partitioned
